# Frame Optimization Analysis

This notebook loads data for a specific frame number and recreates the global optimization scenario for that frame. It includes:
- Loading frame data by frame number
- Recreating the exact optimization scenario from the pipeline
- Per-iteration optimization tracking and visualization
- Comprehensive analysis and visualizations similar to the debug notebook


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R
from easydict import EasyDict as edict
import gtsam
from typing import List, Dict, Tuple, Optional

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Import pipeline components
from point2pose.pipeline.components.local_optimizer import LocalOptimizer
from point2pose.pipeline.components.key_frame_graph import KeyFrameGraph
from point2pose.data_types.object_frame_data import ObjectFrameData
from point2pose.data_types.key_frame import KeyFrame
from point2pose.data_types.frame import Frame
from point2pose.modules.object.object import Object
from point2pose.data_types.point_track_table import PointTrackTable
from point2pose.utils.transform import inverse_SE3, transform_pts

# Matplotlib backend selection
# - Use %matplotlib widget for interactive 3D (rotate/zoom)
# - If it misbehaves in your environment, set USE_MPL_WIDGET=False
USE_MPL_WIDGET = True

try:
    from IPython import get_ipython

    ipython = get_ipython()
    if ipython is not None:
        if USE_MPL_WIDGET:
            try:
                import ipympl  # noqa: F401
                ipython.run_line_magic('matplotlib', 'widget')
                print("Matplotlib backend: widget (interactive)")
            except Exception as e:
                ipython.run_line_magic('matplotlib', 'inline')
                print(f"Matplotlib widget backend unavailable ({e}); using inline")
        else:
            ipython.run_line_magic('matplotlib', 'inline')
            print("Matplotlib backend: inline")
    else:
        import matplotlib

        matplotlib.use('Agg')
        print("Matplotlib backend: Agg (no IPython)")
except Exception as e:
    import matplotlib

    matplotlib.use('Agg')
    print(f"Matplotlib backend fallback: Agg ({e})")


In [ ]:
def find_meta_data_path(
    register_folder: str = None,
    results_dir: Optional[str] = None,
    video_name: Optional[str] = None,
) -> Optional[str]:
    """Find meta_data.npz file in expected locations."""
    script_dir = os.path.dirname(os.path.abspath('.'))
    project_root = os.path.abspath('..')
    
    meta_data_paths = []
    
    # Check new results folder structure first
    if results_dir and video_name:
        new_structure_path = os.path.join(results_dir, video_name, "meta_data", "meta_data.npz")
        meta_data_paths.append(new_structure_path)
        default_results_dir = os.path.join(project_root, "results", "ho3d_single")
        if results_dir != default_results_dir:
            meta_data_paths.append(os.path.join(default_results_dir, video_name, "meta_data", "meta_data.npz"))
    
    # Existing paths for backward compatibility
    if register_folder:
        meta_data_paths.extend([
            os.path.join(register_folder, "meta_data.npz"),
            os.path.join(os.path.dirname(register_folder), "meta_data.npz"),
        ])
    
    for path in meta_data_paths:
        if os.path.exists(path):
            return path
    
    return None

def load_metadata(path):
    """Load metadata from NPZ file."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} does not exist")
    data = np.load(path, allow_pickle=True)
    return data

def get_slice(data, key_prefix, idx, reshape_dim=None):
    """Helper to extract a slice from ragged array storage."""
    try:
        offsets = data[f"{key_prefix}_offsets"]
        lengths = data[f"{key_prefix}_lengths"]
        start = offsets[idx]
        length = lengths[idx]
        arr = data[f"{key_prefix}_data"][start:start+length]
        if reshape_dim:
            if arr.size == 0:
                return arr.reshape(0, reshape_dim)
            return arr.reshape(-1, reshape_dim)
        return arr
    except KeyError:
        return np.array([])

def get_frame_data(data, idx):
    """Extracts ObjectFrameData components for a specific frame index."""
    # 3D points observed in camera frame (from registration)
    cur_3d = get_slice(data, 'reg_curr3d', idx, 3)
    
    # Indices of these points (Track IDs)
    cur_3d_idx = get_slice(data, 'reg_key_points_idx', idx)
    
    # Valid indices used by the registration
    valid_idx = get_slice(data, 'reg_valid_idx', idx)
    if valid_idx.size == 0:
        valid_idx = cur_3d_idx.copy()
    
    # Inliers/Residuals from registration
    inliers = get_slice(data, 'reg_inliers', idx)
    if inliers.size > 0:
        inliers = inliers.astype(bool)
    else:
        inliers = np.ones(cur_3d.shape[0], dtype=bool) if cur_3d.shape[0] > 0 else np.array([], dtype=bool)
        
    residuals = get_slice(data, 'reg_residuals', idx)
    if residuals.size == 0:
        residuals = np.zeros(cur_3d.shape[0]) if cur_3d.shape[0] > 0 else np.array([])
    
    # Uncertainties
    uncertainties = get_slice(data, 'uncertainties', idx)
    if uncertainties.size == 0:
        uncertainties = np.ones(cur_3d.shape[0]) * 0.1 if cur_3d.shape[0] > 0 else np.array([])
    
    # --- New: Visible Points Extraction ---
    # Extract tracks associated with the object
    extract_obj_idx = get_slice(data, 'extract_obj_idx', idx)
    if extract_obj_idx.size == 0:
        extract_obj_idx = valid_idx
        
    # Get all tracks for this frame
    all_tracks = get_slice(data, 'track2d', idx, 2)
    all_visibles = get_slice(data, 'visibles', idx)
    all_uncertainties = get_slice(data, 'uncertainties', idx)
    
    visible_pts_2d = np.zeros((0, 2))
    visible_pts_2d_idx = np.zeros((0,), dtype=int)
    visible_uncertainties = np.zeros((0,))
    
    if extract_obj_idx.size > 0 and all_tracks.shape[0] > 0:
        mask_bounds = extract_obj_idx < all_tracks.shape[0]
        obj_track_indices = extract_obj_idx[mask_bounds]
        
        if obj_track_indices.size > 0:
            if all_visibles.shape[0] > np.max(obj_track_indices):
                obj_visibles = all_visibles[obj_track_indices]
                visible_mask = obj_visibles > 0.5
                
                visible_pts_2d_idx = obj_track_indices[visible_mask]
                visible_pts_2d = all_tracks[visible_pts_2d_idx]
                if all_uncertainties.shape[0] > np.max(visible_pts_2d_idx):
                    visible_uncertainties = all_uncertainties[visible_pts_2d_idx]
                else:
                    visible_uncertainties = np.ones(len(visible_pts_2d_idx)) * 0.1
    
    # Intrinsics
    intrinsics = np.array([[615.377, 0, 320], [0, 615.377, 240], [0, 0, 1]])
    
    # Poses at different stages
    if 'pose_frontend' in data:
        pose_frontend = data['pose_frontend'][idx]
    else:
        print(f"Warning: pose_frontend not found in metadata for frame {idx}, using obj_pose")
        pose_frontend = data['obj_pose'][idx] if 'obj_pose' in data else data['obj_init_pose'][idx]
    
    if 'pose_local' in data:
        pose_local = data['pose_local'][idx]
    else:
        pose_local = pose_frontend

    # Frame ID
    frame_id = int(data['frame_id'][idx])

    # Logged pipeline state (NOT ground truth)
    # - 'obj_pose' in meta_data is the pipeline's current estimated object pose
    obj_pose_logged = data['obj_pose'][idx] if 'obj_pose' in data else None

    # Ground-truth pose (optional): only populated if you load it separately
    gt_pose = None
    if 'gt_pose_by_frame_id' in globals() and gt_pose_by_frame_id is not None:
        gt_pose = gt_pose_by_frame_id.get(frame_id, None)

    # Is keyframe?
    is_keyframe = data['is_key_frame'][idx] if 'is_key_frame' in data else False
    
    return {
        'frame_id': frame_id,
        'pose_frontend': pose_frontend,
        'pose_local': pose_local,
        'obj_pose_logged': obj_pose_logged,
        'gt_pose': gt_pose,
        'cur_3d': cur_3d,
        'cur_3d_idx': cur_3d_idx,
        'valid_idx': valid_idx,
        'inliers': inliers,
        'residuals': residuals,
        'uncertainties': uncertainties,
        'is_keyframe': is_keyframe,
        'intrinsics': intrinsics,
        'visible_pts_2d': visible_pts_2d,
        'visible_pts_2d_idx': visible_pts_2d_idx,
        'visible_uncertainties': visible_uncertainties
    }


In [ ]:
class IterationTrackingOptimizer:
    """Wrapper around LMGraphOptimizer that tracks per-iteration data."""
    
    def __init__(self, base_optimizer):
        self.base_optimizer = base_optimizer
        self.iteration_data = []
        self.iteration_values = []  # gtsam.Values per iteration (for deeper diagnostics)
        # Don't copy graph/values here - they may not be initialized yet
        # We'll copy them in optimize_with_tracking after ensuring base_optimizer has the correct state
        self.graph = None
        self.values = None
        
    def optimize_with_tracking(self, data: ObjectFrameData):
        """Run optimization and track per-iteration changes."""
        self.iteration_data = []
        self.iteration_values = []
        
        # Ensure base_optimizer has graph and values (they should already exist)
        if not hasattr(self.base_optimizer, '_graph'):
            self.base_optimizer._graph = gtsam.NonlinearFactorGraph()
        if not hasattr(self.base_optimizer, '_values'):
            self.base_optimizer._values = gtsam.Values()
        
        # NOW copy the graph and values for tracking (they should be populated by now)
        graph = gtsam.NonlinearFactorGraph(self.base_optimizer._graph)
        values = gtsam.Values(self.base_optimizer._values)
        
        # Store for potential later use
        self.graph = graph
        self.values = values
        
        frame_id = data.frame_id
        Xi = gtsam.symbol("x", frame_id)
        
        # Check if frame is already in graph
        frame_already_added = Xi in self.base_optimizer._inserted_poses
        
        # If frame is already added, update the pose value for re-optimization
        if frame_already_added:
            cur_pose_c2w = inverse_SE3(data.pose)
            cur_pose_c2w_gtsam = gtsam.Pose3(cur_pose_c2w)
            if values.exists(Xi):
                values.update(Xi, cur_pose_c2w_gtsam)
            else:
                values.insert(Xi, cur_pose_c2w_gtsam)
        
        if not frame_already_added:
            # Add the frame to the graph (replicate LMGraphOptimizer logic)
            cur_pose_c2w = inverse_SE3(data.pose)
            cur_pose_c2w_gtsam = gtsam.Pose3(cur_pose_c2w)
            
            # Insert pose variable
            values.insert(Xi, cur_pose_c2w_gtsam)
            self.base_optimizer._inserted_poses.add(Xi)
            
            # Add prior on first pose or between factor
            if not self.base_optimizer._initialized:
                graph.push_back(
                    gtsam.PriorFactorPose3(Xi, cur_pose_c2w_gtsam, self.base_optimizer._prior_noise)
                )
            elif data.rel_pose is not None:
                prev_id = self.base_optimizer._prev_frame_id
                X_prev = gtsam.symbol("x", prev_id)
                rel_T_cim12ci = gtsam.Pose3(inverse_SE3(data.rel_pose))

                # Optional: pose-chain constraint (BetweenFactorPose3)
                # This reduces pose freedom so the map/landmarks are more likely to get corrected.
                use_between = bool(getattr(self.base_optimizer, "_use_between_factor", False))
                if use_between and values.exists(X_prev):
                    between_noise = getattr(self.base_optimizer, "_between_noise", None)

                    # Tuned fallback if the base optimizer doesn't expose a noise model
                    if between_noise is None:
                        base_sigma = (
                            float(max(1e-4, np.mean(data.reg_residuals)))
                            if getattr(data, "reg_residuals", None) is not None
                            and data.reg_residuals.size > 0
                            else 0.05
                        )
                        # Smaller = tighter odometry; tighter odometry tends to push corrections into landmarks.
                        odom_relax_factor = 1.0
                        sigma_between = base_sigma * odom_relax_factor
                        between_noise = gtsam.noiseModel.Diagonal.Sigmas(
                            np.array([sigma_between] * 6, dtype=float)
                        )

                    graph.push_back(
                        gtsam.BetweenFactorPose3(X_prev, Xi, rel_T_cim12ci, between_noise)
                    )
            
            # Insert landmark variables and factors
            if data.reg_inliers.size > 0:
                seed_pose = cur_pose_c2w_gtsam
                if values.exists(Xi):
                    try:
                        seed_pose = values.atPose3(Xi)
                    except RuntimeError:
                        seed_pose = cur_pose_c2w_gtsam
                
                for m, lid in enumerate(data.reg_valid_idx):
                    if not data.reg_inliers[m] or np.isnan(data.reg_cur_3d[m]).any():
                        continue
                    
                    z_cam = data.reg_cur_3d[m]
                    Lj = gtsam.symbol("l", int(lid))
                    
                    # Match LMGraphOptimizer noise scaling
                    # sigma_point = float(max(1e-2, data.reg_residuals[m] ))
                    sigma_point = 1
                    base_noise = gtsam.noiseModel.Diagonal.Sigmas(
                        np.array([0.08*sigma_point, 0.08*sigma_point, 0.01*sigma_point], dtype=float)
                    )
                    point_noise = gtsam.noiseModel.Robust(
                        gtsam.noiseModel.mEstimator.Huber(1.345), base_noise
                    )
                    
                    # Create landmark if missing
                    if Lj not in self.base_optimizer._inserted_landmarks:
                        pw = seed_pose.transformFrom(gtsam.Point3(*z_cam))
                        values.insert(Lj, pw)
                        self.base_optimizer._inserted_landmarks.add(Lj)
                        import bisect
                        bisect.insort(self.base_optimizer.inserted_landmark_ids, int(lid))
                    
                    z_range = float(np.linalg.norm(z_cam))
                    if z_range <= 1e-9:
                        continue
                    
                    z_bearing = gtsam.Unit3(z_cam / z_range)
                    graph.push_back(
                        gtsam.BearingRangeFactor3D(Xi, Lj, z_bearing, z_range, point_noise)
                    )
            
            # Update bookkeeping
            self.base_optimizer._prev_pose_inv = data.pose
            self.base_optimizer._prev_frame_id = frame_id
        
        # Handle first frame (not initialized yet)
        # The first frame gets added with a prior but doesn't get optimized
        if not self.base_optimizer._initialized:
            # For the first frame, we add it but don't optimize yet
            # Record the initial state
            initial_error = graph.error(values) if graph.size() > 0 else 0.0
            initial_pose = None
            if values.exists(Xi):
                initial_pose = values.atPose3(Xi).matrix()
            self.iteration_data.append({
                'iteration': 0,
                'error': initial_error,
                'values': initial_pose,
                'note': 'First frame - no optimization performed, only prior added'
            })
            # Save full Values for later factor/edge diagnostics
            self.iteration_values.append(gtsam.Values(values))

            # Update the base optimizer's state
            self.base_optimizer._graph = graph
            self.base_optimizer._values = values
            self.base_optimizer._initialized = True
            # Return None to indicate no optimization was performed
            # But we still have iteration_data with the initial state
            return None
        
        # Now track iterations
        # Get initial error and pose
        # Check if graph has factors before computing error
        if graph.size() == 0:
            print(f"Warning: Graph is empty after adding frame {frame_id}, cannot optimize")
            return None
            
        initial_error = graph.error(values)
        initial_pose = None
        if values.exists(Xi):
            initial_pose = values.atPose3(Xi).matrix()
        
        self.iteration_data.append({
            'iteration': 0,
            'error': initial_error,
            'values': initial_pose
        })
        # Save full Values for later factor/edge diagnostics
        self.iteration_values.append(gtsam.Values(values))
        
        # Create optimizer for iteration tracking
        lm_params = self.base_optimizer._lm_params
        max_iterations = lm_params.getMaxIterations()
        
        # Check if graph has any factors before optimizing
        if graph.size() == 0:
            print(f"Warning: Graph is empty, cannot optimize frame {frame_id}")
            return None
        
        try:
            optimizer = gtsam.LevenbergMarquardtOptimizer(graph, values, lm_params)
        except RuntimeError as e:
            print(f"Warning: Failed to create optimizer: {e}")
            return None
        
        # Track iterations by manually calling iterate()
        current_values = values
        for i in range(max_iterations):
            try:
                current_error = graph.error(current_values)
                
                # Perform one iteration
                optimizer.iterate()
                new_values = optimizer.values()
                new_error = graph.error(new_values)
                
                # Extract pose if available
                pose_matrix = None
                if new_values.exists(Xi):
                    pose_matrix = new_values.atPose3(Xi).matrix()
                
                error_change = current_error - new_error
                self.iteration_data.append({
                    'iteration': i + 1,
                    'error': new_error,
                    'error_change': error_change,
                    'values': pose_matrix
                })
                # Save full Values for later factor/edge diagnostics
                self.iteration_values.append(gtsam.Values(new_values))
                
                current_values = new_values
                
                # Check convergence
                if current_error > 0 and abs(error_change) < lm_params.getRelativeErrorTol() * current_error:
                    break
                if abs(error_change) < lm_params.getAbsoluteErrorTol():
                    break
                    
            except RuntimeError as e:
                # Optimization converged or failed
                print(f"Warning: Optimization iteration {i+1} failed: {e}")
                break
        
        # Update base optimizer's graph and values with the optimized results
        # Only update if we actually ran optimization (have iteration data)
        if len(self.iteration_data) > 1:  # More than just initial state
            self.base_optimizer._graph = graph  # Update graph (may have new factors)
            self.base_optimizer._values = current_values  # Update values with optimized results
        
        # Extract optimized pose
        if current_values.exists(Xi):
            Xi_hat = current_values.atPose3(Xi)
            pose_opt = inverse_SE3(Xi_hat.matrix())
            
            # Extract landmarks
            num_L = len(self.base_optimizer.inserted_landmark_ids)
            landmark_xyz = np.empty((num_L, 3), dtype=float)
            ids = np.empty((num_L,), dtype=np.int64)
            
            k = 0
            for lid in self.base_optimizer.inserted_landmark_ids:
                Lj = gtsam.symbol("l", int(lid))
                if current_values.exists(Lj):
                    p = np.asarray(current_values.atPoint3(Lj), dtype=float).reshape(3,)
                    landmark_xyz[k, :] = p
                    ids[k] = int(lid)
                    k += 1
            
            landmark_xyz = landmark_xyz[:k, :]
            ids = ids[:k]
            
            from point2pose.data_types.optimizer_result import OptimizerResult
            return OptimizerResult(
                obj_id=data.obj_id,
                frame_id=data.frame_id,
                pose_optimized=pose_opt,
                key_points_optimized=landmark_xyz,
                key_points_idx_optimized=ids,
            )
        
        return None


## Configuration

<!-- Set the frame number and metadata path here. -->


In [ ]:

# data = np.load(meta_data_path, allow_pickle=True)
# keys = sorted(list(meta_data.keys()))
# print(f"Total keys: {len(keys)}")
# for i, key in enumerate(keys, 1):
#     print(f"{i:3d}. {key}")

# for i, is_key_frame in enumerate(meta_data['is_key_frame']):
#     if is_key_frame:
#         print(f"{i}: {is_key_frame}")


In [ ]:
# --- CONFIGURATION ---
FRAME_NUMBER = 440 # Change this to analyze a different frame

# Replay options
# - If False: skip LocalOptimizer.optimize(...) during the replay loop and just use pose_frontend.
USE_LOCAL_OPTIMIZER_IN_REPLAY = False

# Path to metadata file (adjust as needed)
# NOTE: `results/ho3d_single/<video>/meta_data/meta_data.npz` is often not present; `ho3d_all` usually has it.
results_dir = '/home/justin/code/point-to-pose/results/ho3d_single'
video_name = 'MPM10'
meta_data_path = os.path.join(results_dir, video_name, 'meta_data', 'meta_data.npz')

# Fallback paths
if not os.path.exists(meta_data_path):
    print(f"Path {meta_data_path} not found. Trying alternatives...")
    meta_data_path = find_meta_data_path(results_dir=results_dir, video_name=video_name)
    if meta_data_path is None:
        # Try default location
        meta_data_path = '/home/justin/code/point-to-pose/results/ho3d_single/MPM10/meta_data/meta_data.npz'

if not os.path.exists(meta_data_path):
    raise FileNotFoundError(f"Could not find metadata file. Please set meta_data_path manually.")

print(f"Loading metadata from: {meta_data_path}")
meta_data = load_metadata(meta_data_path)
num_frames = len(meta_data['frame_id'])
print(f"Loaded {num_frames} frames.")

# Check if frame exists
frame_ids = meta_data['frame_id']
if FRAME_NUMBER not in frame_ids:
    print(f"Frame {FRAME_NUMBER} not found in metadata.")
    print(f"Available frames: {frame_ids[:20]}..." if len(frame_ids) > 20 else f"Available frames: {frame_ids}")
    raise ValueError(f"Frame {FRAME_NUMBER} not found")

# Find frame index
frame_idx = None
for i, fid in enumerate(frame_ids):
    if fid == FRAME_NUMBER:
        frame_idx = i
        break

print(f"Frame {FRAME_NUMBER} found at index {frame_idx}")

# Check for keyframes in the metadata to help user pick a good frame
if 'is_key_frame' in meta_data:
    keyframe_indices = []
    for i in range(min(frame_idx + 50, len(meta_data['frame_id']))):  # Check up to frame_idx + 50 or end
        if meta_data['is_key_frame'][i]:
            keyframe_indices.append(i)
    if len(keyframe_indices) > 0:
        keyframe_frame_ids_list = [int(meta_data['frame_id'][i]) for i in keyframe_indices]
        print(f"\nFound {len(keyframe_indices)} keyframes in metadata (up to frame {frame_idx + 50}):")
        print(f"  Keyframe indices: {keyframe_indices[:10]}{'...' if len(keyframe_indices) > 10 else ''}")
        print(f"  Keyframe frame IDs: {keyframe_frame_ids_list[:10]}{'...' if len(keyframe_frame_ids_list) > 10 else ''}")

        if FRAME_NUMBER in keyframe_frame_ids_list:
            kf_pos = keyframe_frame_ids_list.index(FRAME_NUMBER)
            if kf_pos == 0:
                print(f"\n⚠️  Frame {FRAME_NUMBER} is the FIRST keyframe - it will only get a prior factor, no optimization iterations.")
                print(
                    f"   To see optimization iterations, try a later keyframe like: {keyframe_frame_ids_list[1] if len(keyframe_frame_ids_list) > 1 else 'a later frame'}"
                )
        else:
            # Not a keyframe: suggest nearby keyframes (prev/next) within the scanned range
            prev_kf_meta_idx = None
            next_kf_meta_idx = None
            for i in reversed(keyframe_indices):
                if i <= frame_idx:
                    prev_kf_meta_idx = i
                    break
            for i in keyframe_indices:
                if i >= frame_idx:
                    next_kf_meta_idx = i
                    break

            print(f"\n⚠️  Frame {FRAME_NUMBER} is NOT a keyframe.")
            if prev_kf_meta_idx is not None:
                print(
                    f"  Prev keyframe: meta idx={prev_kf_meta_idx} -> frame_id={int(meta_data['frame_id'][prev_kf_meta_idx])}"
                )
            if next_kf_meta_idx is not None:
                print(
                    f"  Next keyframe: meta idx={next_kf_meta_idx} -> frame_id={int(meta_data['frame_id'][next_kf_meta_idx])}"
                )
                print(
                    f"  For per-iteration GLOBAL tracking, set FRAME_NUMBER = {int(meta_data['frame_id'][next_kf_meta_idx])}"
                )

# Create config structure matching pipeline
pipeline_cfg = edict({
    'local_optimizer': edict({
        'type': 'lm_graph',
        'params': edict({
            'local_graph_max_num_frames': -1,
            'max_iterations': 20,
            'relative_error_tol': 1e-5,
            'absolute_error_tol': 1e-5,
            'prior_noise_param': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1],

            # --- Map/landmark correction knobs (LMGraphOptimizer) ---
            # Defaults preserve the previous behavior.
            # To encourage the optimizer to correct the map more:
            # - reduce landmark_noise_param (stronger feature constraints)
            # - enable use_between_factor (pose-chain constraint so poses can't freely absorb all error)
            'use_between_factor': False,
            # Pose3 between-noise (tangent space) is 6D: [rot(rad)*3, trans(m)*3]
            # Tuned starting point: ~11deg rot, ~5cm trans
            'between_noise_param': [0.2, 0.2, 0.2, 0.05, 0.05, 0.05],
            'landmark_noise_param': [1.0, 1.0, 1.0],
            'landmark_use_robust': True,
            'landmark_huber_k': 1.345,
            'landmark_sigma_scale_by': 'none',  # 'none' | 'residual' | 'uncertainty'
            'landmark_sigma_min': 1e-6,
        })
    }),
    'global_optimizer': edict({
        'type': 'lm_graph',
        'params': edict({
            'relinearize_threshold': 0.1,
            'relinearize_skip': 1,
            'max_iterations': 20,
            'relative_error_tol': 1e-5,
            'absolute_error_tol': 1e-5,
            'prior_noise_param': [0.01, 0.01, 0.01, 0.01, 0.01, 0.01],

            # --- Map/landmark correction knobs (LMGraphOptimizer) ---
            'use_between_factor': False,
            # Pose3 between-noise (tangent space) is 6D: [rot(rad)*3, trans(m)*3]
            # Tuned starting point: ~11deg rot, ~5cm trans
            'between_noise_param': [0.2, 0.2, 0.2, 0.05, 0.05, 0.05],
            'landmark_noise_param': [1.0, 1.0, 1.0],
            'landmark_use_robust': True,
            'landmark_huber_k': 1.345,
            'landmark_sigma_scale_by': 'none',  # 'none' | 'residual' | 'uncertainty'
            'landmark_sigma_min': 1e-6,
        })
    })
})


## Ground Truth (GT) Loading (Optional, HO3D)

**Important:** the saved `meta_data.npz` does **not** contain ground-truth poses. It contains **estimated** poses like `pose_frontend`, `pose_local`, and `obj_pose` (pipeline state).

If you want real GT error plots, set `HO3D_ROOT` (path to the HO3D dataset root) and run the cell below **before** the replay/analysis cells.

In [ ]:
# Optional: load dataset GT using data readers
# This follows the same idea as scripts/debug_visualization/visualize_pose_debug.py
# (load GT from dataset, aligned to the log's frame_id list).

DATASET = 'ho3d'  # 'ho3d' or 'ycbineoat'
DATA_PATH = '/home/justin/data/HO3D_V3/evaluation/'

# This is used by get_frame_data() if present
# Map: frame_id -> (4,4) GT pose or None
# NOTE: DO NOT confuse meta_data['obj_pose'] with GT (it's pipeline state)
gt_pose_by_frame_id = None

try:
    from point2pose.io.sources.dataset.datareader import Ho3dReader, YcbineoatReader
except Exception as e:
    Ho3dReader = None
    YcbineoatReader = None
    print(f"Datareader import failed: {e}")

if DATA_PATH is None:
    print("DATA_PATH not set -> GT not loaded.")
    print("Set HO3D_ROOT env var (or set DATA_PATH manually) and re-run this cell.")
elif DATASET == 'ho3d' and Ho3dReader is None:
    print("Ho3dReader not available -> GT not loaded")
elif DATASET == 'ycbineoat' and YcbineoatReader is None:
    print("YcbineoatReader not available -> GT not loaded")
else:
    # Determine video path (same conventions as visualize_pose_debug.py)
    if DATASET == 'ho3d':
        video_path = os.path.join(DATA_PATH, 'evaluation', video_name)
        if not os.path.exists(video_path):
            video_path = os.path.join(DATA_PATH, video_name)
        if not os.path.exists(video_path):
            print(f"video_path not found: {video_path}")
            print("Expected HO3D layout: <HO3D_ROOT>/evaluation/<VIDEO_NAME>/rgb/*.jpg")
            video_path = None
    else:
        # ycbineoat layout: <DATA_PATH>/<VIDEO_NAME>
        video_path = os.path.join(DATA_PATH, video_name)
        if not os.path.exists(video_path):
            print(f"video_path not found: {video_path}")
            video_path = None

    if video_path is not None:
        if DATASET == 'ho3d':
            reader = Ho3dReader(video_path, DATA_PATH)
        else:
            reader = YcbineoatReader(video_path)

        print(f"Initialized {type(reader).__name__} with {len(reader)} frames")

        # Optional robustness: map dataset frame_id -> reader list index using id_strs
        fid_to_reader_idx = {}
        if hasattr(reader, 'id_strs') and getattr(reader, 'id_strs') is not None:
            for i, s in enumerate(reader.id_strs):
                try:
                    fid_to_reader_idx[int(s)] = i
                except Exception:
                    pass

        gt_pose_by_frame_id = {}
        num_valid = 0
        for fid in meta_data['frame_id']:
            fid_int = int(fid)
            ridx = fid_to_reader_idx.get(fid_int, fid_int)  # visualize_pose_debug uses idx=fid

            pose = None
            if 0 <= ridx < len(reader):
                pose = reader.get_gt_pose(ridx)

            gt_pose_by_frame_id[fid_int] = pose
            if pose is not None:
                num_valid += 1

        print(f"Loaded dataset GT: {num_valid}/{len(meta_data['frame_id'])} frames have GT")

        # Align GT poses to first frame (make first valid GT pose identity)
        # This matches the alignment in run_ho3d_single.py
        from point2pose.utils.transform import inverse_SE3

        # Find first valid GT pose (by frame_id order)
        first_valid_gt = None
        first_valid_fid = None
        for fid in meta_data['frame_id']:
            fid_int = int(fid)
            gt_pose = gt_pose_by_frame_id.get(fid_int, None)
            if gt_pose is not None:
                first_valid_gt = np.asarray(gt_pose, dtype=float)
                first_valid_fid = fid_int
                break

        if first_valid_gt is not None:
            # Normalize GT to the first valid frame so that the first GT pose becomes identity.
            # We do: GT_rel = GT @ inv(GT0)
            T_align = inverse_SE3(first_valid_gt)

            # Apply alignment to all GT poses
            aligned_count = 0
            for fid_int, gt_pose in gt_pose_by_frame_id.items():
                if gt_pose is not None:
                    gt_pose_by_frame_id[fid_int] = np.asarray(gt_pose, dtype=float) @ T_align
                    aligned_count += 1

            print(
                f"Aligned {aligned_count} GT poses to first frame (frame_id={first_valid_fid} is now identity)"
            )
        else:
            print("No valid GT poses found - skipping alignment")

        print("Now re-run the replay cell so traj_gt gets populated.")


## Recreate Optimization Scenario

This section recreates the optimization scenario up to the target frame, building the graph incrementally as the pipeline does.


In [ ]:
# Initialize pipeline components
local_optimizer = LocalOptimizer(pipeline_cfg)
kf_graph = KeyFrameGraph(pipeline_cfg)

# Create mock objects for state tracking
mock_obj = Object(0)
mock_obj.pose = np.eye(4)
mock_track_table = PointTrackTable.new(n0=0)
mock_track_table.obj2track_map = {0: []}

# Storage for trajectories
# NOTE on provenance:
# - *_logged come directly from meta_data logs
# - traj_local / traj_global are recomputed ("rerun") in this notebook
traj_frontend = []         # LOGGED: meta_data['pose_frontend']
traj_local_logged = []     # LOGGED: meta_data['pose_local']
traj_obj_pose_logged = []  # LOGGED: meta_data['obj_pose'] (pipeline state after the step)

traj_local = []            # RERUN: local_optimizer.optimize(...) starting from pose_frontend
traj_global = []           # RERUN: kf_graph.update(...) on keyframes

traj_gt = []               # OPTIONAL external GT (None if not loaded)

# Keyframe buffering (fed into global optimizer)
keyframes_list = []
kf_idx_counter = 0
keyframe_frame_ids = []

# --- Diagnostics: keyframe poses before/after global optimization ---
# Indexed by keyframe index (kf_idx)
kf_frame_id_by_kf_idx = {}
kf_num_obs_by_kf_idx = {}
kf_pose_pre_global = {}
kf_pose_post_global = {}
kf_gt_pose_by_kf_idx = {}

# Global graph objective trend (per keyframe update)
global_obj_error_before = []
global_obj_error_after = []
global_graph_num_factors = []
global_graph_num_values = []
global_kf_idx_updates = []

# --- Landmarks snapshots after each GLOBAL keyframe optimization ---
# Each entry corresponds to one kf_graph.update(...) call.
# Stored as: {'obj_id': int, 'kf_idx': int, 'frame_id': int, 'xyz': (N,3), 'ids': (N,)}
global_landmarks_updates = []

# Recreate pipeline up to (but not including) target frame
# We'll track iterations when we add the target frame
print(f"Recreating pipeline optimization up to (but not including) frame {FRAME_NUMBER}...")

for i in range(frame_idx):  # Stop before target frame
    fd = get_frame_data(meta_data, i)
    
    # Compute relative pose
    if i == 0:
        rel_pose = np.eye(4)
    else:
        prev_fd = get_frame_data(meta_data, i-1)
        prev_pose = prev_fd['pose_frontend']
        cur_pose = fd['pose_frontend']
        prev_pose_inv = inverse_SE3(prev_pose)
        rel_pose = cur_pose @ prev_pose_inv
    
    # Store logged poses
    traj_frontend.append(fd['pose_frontend'].copy())
    traj_local_logged.append(fd['pose_local'].copy() if fd.get('pose_local', None) is not None else None)
    traj_obj_pose_logged.append(fd['obj_pose_logged'].copy() if fd.get('obj_pose_logged', None) is not None else None)
    
    # Local optimization (optional)
    object_frame_data = ObjectFrameData(
        obj_id=0,
        frame_id=fd['frame_id'],
        intrinsics=fd['intrinsics'],
        pose=fd['pose_frontend'],
        rel_pose=rel_pose,
        visible_pts_2d=fd["visible_pts_2d"],
        visible_pts_2d_idx=fd["visible_pts_2d_idx"],
        visible_uncertainties=fd["visible_uncertainties"],
        reg_cur_3d=fd['cur_3d'],
        reg_cur_3d_idx=fd['cur_3d_idx'],
        reg_valid_idx=fd['valid_idx'],
        reg_inliers=fd['inliers'],
        reg_residuals=fd['residuals'],
        reg_uncertainties=fd['uncertainties']
    )

    _use_local = bool(globals().get('USE_LOCAL_OPTIMIZER_IN_REPLAY', True))
    opt_result = local_optimizer.optimize(object_frame_data) if _use_local else None

    if opt_result is not None:
        mock_obj.pose = opt_result.pose_optimized.copy()
        if len(fd['cur_3d_idx']) > 0:
            mock_track_table.obj2track_map[0] = fd['cur_3d_idx'].tolist()
        traj_local.append(mock_obj.pose.copy())
    else:
        mock_obj.pose = fd['pose_frontend'].copy()
        traj_local.append(fd['pose_frontend'].copy())

    # Keyframe handling
    if fd['is_keyframe']:
        keyframe_frame_ids.append(fd['frame_id'])
        
        # Use cur_3d_idx length as the definitive size for observation arrays
        # This ensures all observation arrays have consistent sizes
        cur_3d_idx = fd['cur_3d_idx']
        num_obs = len(cur_3d_idx)  # Use cur_3d_idx length as definitive size
        
        # Ensure all arrays match num_obs size
        cur_3d = fd['cur_3d']
        if len(cur_3d) != num_obs:
            if len(cur_3d) > num_obs:
                cur_3d = cur_3d[:num_obs]
            elif len(cur_3d) < num_obs and num_obs > 0:
                padding = np.zeros((num_obs - len(cur_3d), 3))
                cur_3d = np.vstack([cur_3d, padding]) if len(cur_3d) > 0 else padding
        
        inliers = fd['inliers']
        if len(inliers) != num_obs:
            if len(inliers) > num_obs:
                inliers = inliers[:num_obs]
            elif num_obs > 0:
                inliers = np.pad(inliers, (0, num_obs - len(inliers)), constant_values=True)
        
        residuals = fd['residuals']
        if len(residuals) != num_obs:
            if len(residuals) > num_obs:
                residuals = residuals[:num_obs]
            elif num_obs > 0:
                residuals = np.pad(residuals, (0, num_obs - len(residuals)), constant_values=0.0)
        
        valid_idx = fd['valid_idx']
        if len(valid_idx) != num_obs:
            if len(valid_idx) > num_obs:
                valid_idx = valid_idx[:num_obs]
            elif num_obs > 0:
                pad_value = valid_idx[-1] if len(valid_idx) > 0 else 0
                valid_idx = np.pad(valid_idx, (0, num_obs - len(valid_idx)), constant_values=pad_value)
        
        # Get uncertainties - ensure it matches num_obs
        if num_obs > 0 and len(fd['uncertainties']) > 0:
            try:
                max_idx = np.max(cur_3d_idx) if len(cur_3d_idx) > 0 else -1
                if max_idx < len(fd['uncertainties']):
                    obs_uncertainties = fd['uncertainties'][cur_3d_idx]
                else:
                    obs_uncertainties = fd['uncertainties'][:num_obs] if len(fd['uncertainties']) >= num_obs else np.pad(fd['uncertainties'], (0, num_obs - len(fd['uncertainties'])), constant_values=0.1)
            except:
                obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
        else:
            obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1 if num_obs > 0 else np.array([], dtype=float)
        
        
        frame_obj = Frame(
            id=fd['frame_id'],
            rgb=np.zeros((480, 640, 3), dtype=np.uint8),  # Dummy RGB
            intrinsics=fd['intrinsics']
        )
        
        kf = KeyFrame(
            frame_id=fd['frame_id'],
            obj_id=0,
            kf_idx=kf_idx_counter,
            timestamp=None,
            frame=frame_obj,
            pose=mock_obj.pose.copy(),
            kp_track_indices=cur_3d_idx.copy(),
            kp_2d=np.zeros((num_obs, 2)),
            kp_3d_camera=cur_3d.copy(),
            kp_3d_object=np.zeros((num_obs, 3)),
            kp_valid=np.ones(num_obs, dtype=bool),
            obs_track_indices=cur_3d_idx.copy(),
            obs_2d=np.zeros((num_obs, 2)),
            obs_3d_camera=cur_3d.copy(),
            obs_3d_object=np.zeros((num_obs, 3)),
            obs_valid=np.ones(num_obs, dtype=bool),
            obs_visible=np.ones(num_obs, dtype=bool),
            obs_uncertainties=obs_uncertainties,
            reg_correspond_curr3d=cur_3d.copy(),
            reg_inliers=inliers,
            reg_residuals=residuals,
            reg_valid_idx=valid_idx,
            dense_pts=np.zeros((0, 3))
        )

        # Diagnostics: store the pre-global pose/GT for this keyframe index
        kf_frame_id_by_kf_idx[kf_idx_counter] = int(fd['frame_id'])
        kf_num_obs_by_kf_idx[kf_idx_counter] = int(num_obs)
        kf_pose_pre_global[kf_idx_counter] = kf.pose.copy()
        kf_gt_pose_by_kf_idx[kf_idx_counter] = fd['gt_pose'].copy() if fd['gt_pose'] is not None else None

        keyframes_list.append(kf)
        kf_idx_counter += 1
        
        # Reset local optimizer on keyframe (if enabled)
        if bool(globals().get('USE_LOCAL_OPTIMIZER_IN_REPLAY', True)):
            local_optimizer.reset(0)
    
    # Global optimization (only for keyframes)
    if keyframes_list and fd['is_keyframe']:
        # Get optimizer before update to verify it's the same instance
        opt_before = kf_graph._get_optimizer(0)
        opt_before_id = id(opt_before)
        opt_before_graph_size = opt_before._graph.size() if hasattr(opt_before, '_graph') else 0

        # Objective (before adding this batch)
        if hasattr(opt_before, '_graph') and hasattr(opt_before, '_values') and opt_before._graph.size() > 0:
            obj_before = float(opt_before._graph.error(opt_before._values))
        else:
            obj_before = 0.0

        updated_poses, updated_landmarks = kf_graph.update(keyframes_list)

        # Snapshot optimized landmarks after this keyframe update (for later visualization)
        try:
            _kf_last = keyframes_list[-1]
            _obj_id = int(_kf_last.obj_id)
            _kf_idx = int(_kf_last.kf_idx)
            _frame_id = int(_kf_last.frame_id)

            _xyz = None
            _ids = None

            if isinstance(updated_landmarks, dict) and _obj_id in updated_landmarks:
                _xyz, _ids = updated_landmarks[_obj_id]
                _xyz = np.asarray(_xyz, dtype=float)
                _ids = np.asarray(_ids, dtype=int).reshape(-1)
            else:
                # Fallback: extract from underlying optimizer state (covers first-KF no-opt case)
                _opt_tmp = kf_graph._get_optimizer(_obj_id)
                if hasattr(_opt_tmp, "_values") and hasattr(_opt_tmp, "inserted_landmark_ids"):
                    _vals_tmp = _opt_tmp._values
                    _all_ids = np.asarray(_opt_tmp.inserted_landmark_ids, dtype=int).reshape(-1)
                    _xyz_tmp = np.full((_all_ids.shape[0], 3), np.nan, dtype=float)
                    for _j, _lid in enumerate(_all_ids.tolist()):
                        Lj = gtsam.symbol("l", int(_lid))
                        if _vals_tmp.exists(Lj):
                            _xyz_tmp[_j] = np.asarray(_vals_tmp.atPoint3(Lj), dtype=float).reshape(3,)
                    _m = np.isfinite(_xyz_tmp).all(axis=1)
                    _xyz = _xyz_tmp[_m]
                    _ids = _all_ids[_m]

            if _xyz is not None and _ids is not None and _xyz.size > 0:
                # Store pose/GT + per-keyframe measurement arrays so we can compute landmark errors later
                _pose_est = np.asarray(getattr(_kf_last, "pose", np.eye(4)), dtype=float).copy()
                _pose_gt = kf_gt_pose_by_kf_idx.get(_kf_idx, None)
                if _pose_gt is not None:
                    _pose_gt = np.asarray(_pose_gt, dtype=float).copy()

                _meas_xyz_cam = np.asarray(
                    getattr(_kf_last, "reg_correspond_curr3d", np.zeros((0, 3))), dtype=float
                )
                if _meas_xyz_cam.ndim == 1 and _meas_xyz_cam.size == 0:
                    _meas_xyz_cam = np.zeros((0, 3), dtype=float)
                if _meas_xyz_cam.ndim == 2 and _meas_xyz_cam.shape[1] != 3:
                    # Best effort reshape
                    _meas_xyz_cam = _meas_xyz_cam.reshape(-1, 3)

                _meas_ids = np.asarray(
                    getattr(_kf_last, "reg_valid_idx", np.zeros((0,), dtype=int)), dtype=int
                ).reshape(-1)
                _meas_inliers = np.asarray(
                    getattr(
                        _kf_last,
                        "reg_inliers",
                        np.ones((_meas_ids.shape[0],), dtype=bool),
                    ),
                    dtype=bool,
                ).reshape(-1)

                _min = int(min(_meas_xyz_cam.shape[0], _meas_ids.shape[0], _meas_inliers.shape[0]))
                _meas_xyz_cam = _meas_xyz_cam[:_min]
                _meas_ids = _meas_ids[:_min]
                _meas_inliers = _meas_inliers[:_min]

                global_landmarks_updates.append(
                    {
                        "obj_id": _obj_id,
                        "kf_idx": _kf_idx,
                        "frame_id": _frame_id,
                        "xyz": _xyz,
                        "ids": _ids,
                        "pose_est": _pose_est,
                        "pose_gt": _pose_gt,
                        "meas_xyz_cam": _meas_xyz_cam,
                        "meas_ids": _meas_ids,
                        "meas_inliers": _meas_inliers,
                    }
                )
        except Exception as _e:
            print(f"  [Frame {fd['frame_id']}] Warning: failed to snapshot landmarks: {_e}")

        # Verify optimizer after update
        opt_after = kf_graph._get_optimizer(0)
        opt_after_id = id(opt_after)
        opt_after_graph_size = opt_after._graph.size() if hasattr(opt_after, '_graph') else 0

        # Objective (after optimization)
        if hasattr(opt_after, '_graph') and hasattr(opt_after, '_values') and opt_after._graph.size() > 0:
            obj_after = float(opt_after._graph.error(opt_after._values))
            num_values_after = int(opt_after._values.size())
        else:
            obj_after = 0.0
            num_values_after = 0

        # Record global trend (one entry per update call)
        global_obj_error_before.append(obj_before)
        global_obj_error_after.append(obj_after)
        global_graph_num_factors.append(int(opt_after_graph_size))
        global_graph_num_values.append(num_values_after)
        global_kf_idx_updates.append(int(keyframes_list[-1].kf_idx))

        # Debug: Check if optimizer instance persisted
        if opt_before_id == opt_after_id:
            print(f"  [Frame {fd['frame_id']}] Optimizer instance persisted (id={opt_before_id}), graph size: {opt_before_graph_size} -> {opt_after_graph_size}")
        else:
            print(f"  [Frame {fd['frame_id']}] WARNING: Optimizer instance changed! Before: {opt_before_id}, After: {opt_after_id}")

        for kf in keyframes_list:
            # Store post-global pose for this keyframe (even if LM was skipped)
            kf_pose_post_global[kf.kf_idx] = np.asarray(kf.pose, dtype=float).copy()

            key = (kf.obj_id, kf.kf_idx)
            if key in updated_poses:
                mock_obj.pose = updated_poses[key].copy()
            else:
                # If optimization was skipped (e.g., first keyframe), keep the incoming pose
                mock_obj.pose = np.asarray(kf.pose, dtype=float).copy()
        keyframes_list = []
    
    traj_global.append(mock_obj.pose.copy())
    
    # Store GT (optional; None if not loaded)
    traj_gt.append(fd['gt_pose'].copy() if fd.get('gt_pose', None) is not None else None)

print(f"Completed optimization loop. Processed {frame_idx} frames (up to but not including frame {FRAME_NUMBER}).")
print(f"Found {kf_idx_counter} keyframes before frame {FRAME_NUMBER}.")
if len(keyframe_frame_ids) > 0:
    print(f"Keyframe Frame IDs before target: {keyframe_frame_ids}")

print("\n=== Trajectory provenance ===")
print("traj_frontend:       LOGGED meta_data['pose_frontend']")
print("traj_local_logged:   LOGGED meta_data['pose_local']")
print("traj_obj_pose_logged:LOGGED meta_data['obj_pose'] (pipeline state; can include global updates)")
print("traj_local:          RERUN LocalOptimizer.optimize(...)")
print("traj_global:         RERUN KeyFrameGraph.update(...)")
print("traj_gt:             External GT (loaded separately) or None")


## Per-Iteration Optimization Analysis

Now we'll recreate the optimization for the target frame with per-iteration tracking.


In [ ]:
# Get frame data for target frame
target_fd = get_frame_data(meta_data, frame_idx)

# Check if this is a keyframe
is_target_keyframe = target_fd['is_keyframe']
print(f"\nFrame {FRAME_NUMBER} is {'a KEYFRAME' if is_target_keyframe else 'NOT a keyframe'}")

# Now we'll add the target frame and track iterations
# The graph is built up to (but not including) the target frame

# Select the appropriate optimizer
if is_target_keyframe:
    # For keyframes, we need to add to global optimizer
    # Get the optimizer (it should already exist if keyframes were processed)
    tracking_opt = kf_graph._get_optimizer(0)
    
    # Debug: Check the optimizer state BEFORE copying
    print(f"\n=== Optimizer State Check ===")
    print(f"tracking_opt type: {type(tracking_opt)}")
    print(f"tracking_opt id: {id(tracking_opt)}")
    print(f"Has _graph attribute: {hasattr(tracking_opt, '_graph')}")
    print(f"Has _values attribute: {hasattr(tracking_opt, '_values')}")
    
    if hasattr(tracking_opt, '_graph'):
        print(f"Graph size: {tracking_opt._graph.size()}")
        print(f"Graph type: {type(tracking_opt._graph)}")
    else:
        print("ERROR: tracking_opt does not have _graph attribute!")
        
    if hasattr(tracking_opt, '_values'):
        print(f"Values size: {tracking_opt._values.size()}")
        print(f"Values type: {type(tracking_opt._values)}")
    else:
        print("ERROR: tracking_opt does not have _values attribute!")
    
    if hasattr(tracking_opt, '_inserted_poses'):
        print(f"Inserted poses: {len(tracking_opt._inserted_poses)}")
        if len(tracking_opt._inserted_poses) > 0:
            print(f"  Pose symbols: {sorted([int(gtsam.Symbol(s).index()) for s in tracking_opt._inserted_poses])}")
    else:
        print("ERROR: tracking_opt does not have _inserted_poses attribute!")
    
    if hasattr(tracking_opt, '_initialized'):
        print(f"Initialized: {tracking_opt._initialized}")
    else:
        print("ERROR: tracking_opt does not have _initialized attribute!")
    
    # Check if optimizer was actually used (has graph/values)
    # If not, it means no keyframes were processed before this one
    if not hasattr(tracking_opt, '_graph') or tracking_opt._graph.size() == 0:
        print(f"\n⚠️  Warning: Global optimizer is empty. Frame {FRAME_NUMBER} appears to be the first keyframe.")
        print("The optimizer will be initialized when we add this frame.")
    else:
        print(f"\n✓ Optimizer has graph with {tracking_opt._graph.size()} factors and {tracking_opt._values.size()} values")
    
    # Create KeyFrame for target frame
    # Use cur_3d_idx length as the definitive size for observation arrays (avoids boolean-index mismatches)
    cur_3d_idx = target_fd['cur_3d_idx']
    num_obs = len(cur_3d_idx)

    # Ensure all arrays match num_obs size (pad/clip)
    cur_3d = target_fd['cur_3d']
    if len(cur_3d) != num_obs:
        if len(cur_3d) > num_obs:
            cur_3d = cur_3d[:num_obs]
        elif len(cur_3d) < num_obs and num_obs > 0:
            padding = np.zeros((num_obs - len(cur_3d), 3))
            cur_3d = np.vstack([cur_3d, padding]) if len(cur_3d) > 0 else padding

    inliers = target_fd['inliers']
    if len(inliers) != num_obs:
        if len(inliers) > num_obs:
            inliers = inliers[:num_obs]
        elif num_obs > 0:
            inliers = np.pad(inliers, (0, num_obs - len(inliers)), constant_values=True)

    residuals = target_fd['residuals']
    if len(residuals) != num_obs:
        if len(residuals) > num_obs:
            residuals = residuals[:num_obs]
        elif num_obs > 0:
            residuals = np.pad(residuals, (0, num_obs - len(residuals)), constant_values=0.0)

    valid_idx = target_fd['valid_idx']
    if len(valid_idx) != num_obs:
        if len(valid_idx) > num_obs:
            valid_idx = valid_idx[:num_obs]
        elif num_obs > 0:
            pad_value = valid_idx[-1] if len(valid_idx) > 0 else 0
            valid_idx = np.pad(valid_idx, (0, num_obs - len(valid_idx)), constant_values=pad_value)

    # Get uncertainties - ensure it matches num_obs
    if num_obs > 0 and len(target_fd['uncertainties']) > 0:
        try:
            max_idx = np.max(cur_3d_idx) if len(cur_3d_idx) > 0 else -1
            if max_idx < len(target_fd['uncertainties']):
                obs_uncertainties = target_fd['uncertainties'][cur_3d_idx]
            else:
                obs_uncertainties = target_fd['uncertainties'][:num_obs] if len(target_fd['uncertainties']) >= num_obs else np.pad(target_fd['uncertainties'], (0, num_obs - len(target_fd['uncertainties'])), constant_values=0.1)
        except:
            obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
    else:
        obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1 if num_obs > 0 else np.array([], dtype=float)

    # Use the pose from local optimization (or frontend if not optimized yet)
    # Create minimal Frame object
    frame_obj = Frame(
        id=target_fd['frame_id'],
        rgb=np.zeros((480, 640, 3), dtype=np.uint8),  # Dummy RGB
        intrinsics=target_fd['intrinsics']
    )

    target_kf = KeyFrame(
        frame_id=target_fd['frame_id'],
        obj_id=0,
        kf_idx=kf_idx_counter,
        timestamp=None,
        frame=frame_obj,
        pose=mock_obj.pose.copy(),  # Current pose from previous frames
        kp_track_indices=cur_3d_idx.copy(),
        kp_2d=np.zeros((num_obs, 2)),
        kp_3d_camera=cur_3d.copy(),
        kp_3d_object=np.zeros((num_obs, 3)),
        kp_valid=np.ones(num_obs, dtype=bool),
        obs_track_indices=cur_3d_idx.copy(),
        obs_2d=np.zeros((num_obs, 2)),
        obs_3d_camera=cur_3d.copy(),
        obs_3d_object=np.zeros((num_obs, 3)),
        obs_valid=np.ones(num_obs, dtype=bool),
        obs_visible=np.ones(num_obs, dtype=bool),
        obs_uncertainties=obs_uncertainties,
        reg_correspond_curr3d=cur_3d.copy(),
        reg_inliers=inliers,
        reg_residuals=residuals,
        reg_valid_idx=valid_idx,
        dense_pts=np.zeros((0, 3))
    )
    
    # For keyframes, we need to manually track the optimization
    # The kf_graph.update() method calls the optimizer internally
    # We'll need to intercept that call or manually call the optimizer
    
    # Recreate ObjectFrameData as kf_graph.update() would create it
    obj_id = target_kf.obj_id
    kf_idx = target_kf.kf_idx
    cur_pose = np.asarray(target_kf.pose, dtype=float)
    
    # Compute relative pose
    if obj_id in kf_graph._last_kf_pose:
        prev_pose = kf_graph._last_kf_pose[obj_id]
        prev_pose_inv = inverse_SE3(prev_pose)
        rel_pose = cur_pose @ prev_pose_inv
    else:
        rel_pose = np.eye(4, dtype=float)
    
    # Build measurement data
    if (target_kf.obs_3d_camera is not None and target_kf.obs_3d_camera.size > 0 and
        target_kf.obs_valid is not None and target_kf.obs_visible is not None):
        valid_mask = np.asarray(target_kf.obs_valid, dtype=bool) & np.asarray(target_kf.obs_visible, dtype=bool)
        if np.any(valid_mask):
            cur_3d = np.asarray(target_kf.obs_3d_camera[valid_mask], dtype=float)
            cur_3d_idx = np.asarray(target_kf.obs_track_indices[valid_mask], dtype=int)
            if target_kf.obs_uncertainties is not None:
                uncertainties = np.asarray(target_kf.obs_uncertainties[valid_mask], dtype=float)
            else:
                uncertainties = 0.5 * np.ones((cur_3d.shape[0],), dtype=float)
        else:
            cur_3d = np.zeros((0, 3), dtype=float)
            cur_3d_idx = np.zeros((0,), dtype=int)
            uncertainties = np.zeros((0,), dtype=float)
    else:
        cur_3d = np.zeros((0, 3), dtype=float)
        cur_3d_idx = np.zeros((0,), dtype=int)
        uncertainties = np.zeros((0,), dtype=float)
    
    target_object_frame_data = ObjectFrameData(
        obj_id=obj_id,
        frame_id=kf_idx,
        pose=cur_pose,
        intrinsics=target_fd["intrinsics"],
        visible_pts_2d=target_fd["visible_pts_2d"],
        visible_pts_2d_idx=target_fd["visible_pts_2d_idx"],
        visible_uncertainties=target_fd["visible_uncertainties"],
        rel_pose=rel_pose,
        reg_cur_3d=cur_3d,
        reg_cur_3d_idx=cur_3d_idx,
        reg_valid_idx=target_kf.reg_valid_idx,
        reg_inliers=target_kf.reg_inliers,
        reg_residuals=target_kf.reg_residuals,
        reg_uncertainties=uncertainties,
    )

# This per-iteration tracker mirrors the GLOBAL keyframe graph update (KeyFrameGraph.update -> LMGraphOptimizer.optimize).
# Global updates only happen on keyframes.
if not is_target_keyframe:
    print(
        "\nSelected FRAME_NUMBER is NOT a keyframe -> there is no global keyframe-graph update to track.\n"
        "Pick a keyframe frame_id and re-run: (1) config cell, (2) replay cell, (3) this cell."
    )

    # Suggest nearby keyframes (by metadata index) if available
    if 'is_key_frame' in meta_data:
        try:
            import bisect

            kf_inds = np.where(np.asarray(meta_data['is_key_frame']).astype(bool))[0].astype(int).tolist()
            if len(kf_inds) > 0:
                pos = bisect.bisect_left(kf_inds, int(frame_idx))
                cand_meta_idx = []
                for j in [pos - 2, pos - 1, pos, pos + 1, pos + 2]:
                    if 0 <= j < len(kf_inds):
                        cand_meta_idx.append(kf_inds[j])

                # de-dup (keep order)
                cand_meta_idx = list(dict.fromkeys(cand_meta_idx))

                if len(cand_meta_idx) > 0:
                    print("\nNearby keyframes (meta idx -> frame_id):")
                    for ii in cand_meta_idx:
                        print(f"  - {ii} -> {int(meta_data['frame_id'][ii])}")

                    # If the first suggestion is the very first keyframe, also suggest the next
                    first_kf_meta_idx = int(min(kf_inds))
                    if cand_meta_idx[0] == first_kf_meta_idx and len(kf_inds) > 1:
                        print(
                            f"\nNote: frame_id={int(meta_data['frame_id'][cand_meta_idx[0]])} is the FIRST keyframe (only prior, no iterations).\n"
                            f"Try a later keyframe like: frame_id={int(meta_data['frame_id'][kf_inds[1]])}"
                        )
                    else:
                        print(
                            f"\nTry setting FRAME_NUMBER = {int(meta_data['frame_id'][cand_meta_idx[0]])} (and re-run config/replay/this cell)."
                        )
        except Exception as e:
            print(f"(Could not compute nearby keyframe suggestions: {e})")

    raise RuntimeError("FRAME_NUMBER is not a keyframe (no global optimizer state to track)")

iteration_tracker = IterationTrackingOptimizer(tracking_opt)


# ALWAYS copy the state from tracking_opt to iteration_tracker.base_optimizer
# This ensures we're working with the same state, even if empty
print(f"\nCopying optimizer state from kf_graph optimizer...")

# Since base_optimizer IS tracking_opt, they share the same state
# Just verify the state exists and show it
if hasattr(tracking_opt, '_graph'):
    print(f"  Graph size: {tracking_opt._graph.size()}")
    print(f"  Graph type: {type(tracking_opt._graph)}")
else:
    print("  ERROR: tracking_opt does not have _graph attribute!")
    tracking_opt._graph = gtsam.NonlinearFactorGraph()

if hasattr(tracking_opt, '_values'):
    print(f"  Values size: {tracking_opt._values.size()}")
    print(f"  Values type: {type(tracking_opt._values)}")
else:
    print("  ERROR: tracking_opt does not have _values attribute!")
    tracking_opt._values = gtsam.Values()

if hasattr(tracking_opt, '_inserted_poses'):
    print(f"  Inserted poses: {len(tracking_opt._inserted_poses)}")
    if len(tracking_opt._inserted_poses) > 0:
        print(f"    Pose symbols: {sorted([int(gtsam.Symbol(s).index()) for s in tracking_opt._inserted_poses])}")
else:
    print("  ERROR: tracking_opt does not have _inserted_poses attribute!")
    tracking_opt._inserted_poses = set()

if hasattr(tracking_opt, '_initialized'):
    print(f"  Initialized: {tracking_opt._initialized}")
else:
    print("  ERROR: tracking_opt does not have _initialized attribute!")
    tracking_opt._initialized = False

if hasattr(tracking_opt, '_prior_noise'):
    print(f"  Has _prior_noise: ✓")
else:
    print("  ERROR: tracking_opt does not have _prior_noise attribute!")
    # Initialize from config
    if is_target_keyframe:
        prior_noise_param = pipeline_cfg.global_optimizer.params.prior_noise_param
    else:
        prior_noise_param = pipeline_cfg.local_optimizer.params.prior_noise_param
    tracking_opt._prior_noise = gtsam.noiseModel.Diagonal.Sigmas(
        np.array(prior_noise_param, dtype=float)
    )

if hasattr(tracking_opt, '_lm_params'):
    print(f"  Has _lm_params: ✓")
else:
    print("  ERROR: tracking_opt does not have _lm_params attribute!")
    # Initialize from config
    tracking_opt._lm_params = gtsam.LevenbergMarquardtParams()
    if is_target_keyframe:
        cfg_params = pipeline_cfg.global_optimizer.params
    else:
        cfg_params = pipeline_cfg.local_optimizer.params
    tracking_opt._lm_params.setMaxIterations(cfg_params.max_iterations)
    tracking_opt._lm_params.setRelativeErrorTol(cfg_params.relative_error_tol)
    tracking_opt._lm_params.setAbsoluteErrorTol(cfg_params.absolute_error_tol)
    tracking_opt._lm_params.setlambdaInitial(1e-1)
    tracking_opt._lm_params.setVerbosityLM("SUMMARY")

print(f"\n✓ Optimizer state verified. Graph and values are ready for tracking.")

# Run optimization with tracking
print(f"\nRunning optimization with per-iteration tracking for frame {FRAME_NUMBER}...")
print("(This will add the frame to the graph and track each iteration)")

# Debug: Check optimizer state before tracking
print(f"Debug: Optimizer initialized: {hasattr(tracking_opt, '_initialized') and tracking_opt._initialized}")
print(f"Debug: Graph size: {tracking_opt._graph.size() if hasattr(tracking_opt, '_graph') else 'N/A'}")
print(f"Debug: Values size: {tracking_opt._values.size() if hasattr(tracking_opt, '_values') else 'N/A'}")
if hasattr(tracking_opt, '_inserted_poses'):
    print(f"Debug: Inserted poses: {len(tracking_opt._inserted_poses)}")
    if len(tracking_opt._inserted_poses) > 0:
        print(f"Debug: Inserted pose IDs: {sorted([int(gtsam.Symbol(s).index()) for s in tracking_opt._inserted_poses])}")
print(f"Debug: Frame ID in ObjectFrameData: {target_object_frame_data.frame_id}")
if hasattr(tracking_opt, '_inserted_poses'):
    Xi_check = gtsam.symbol("x", target_object_frame_data.frame_id)
    print(f"Debug: Frame already in graph: {Xi_check in tracking_opt._inserted_poses}")
    
# Additional debug for keyframes
if is_target_keyframe:
    print(f"Debug: Number of keyframes processed before target: {kf_idx_counter}")
    print(f"Debug: Previous keyframe frame IDs: {keyframe_frame_ids}")

# Call the iteration tracker
opt_result = iteration_tracker.optimize_with_tracking(target_object_frame_data)

# Extract iteration data
iteration_errors = [d['error'] for d in iteration_tracker.iteration_data]
iteration_changes = [d.get('error_change', 0) for d in iteration_tracker.iteration_data]
iteration_poses = [d.get('values') for d in iteration_tracker.iteration_data]

# Also store a snapshot of the FINAL optimized landmarks for this keyframe (for keyframe-to-keyframe landmark viz)
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    global_landmarks_updates = []

if opt_result is not None and getattr(opt_result, "key_points_optimized", None) is not None:
    try:
        _obj_id = int(getattr(opt_result, "obj_id", 0))
        _kf_idx = int(getattr(target_object_frame_data, "frame_id", -1))
        _frame_id = int(FRAME_NUMBER)

        _xyz = np.asarray(opt_result.key_points_optimized, dtype=float)
        _ids = np.asarray(getattr(opt_result, "key_points_idx_optimized", []), dtype=int).reshape(-1)

        if _xyz.ndim != 2 or _xyz.shape[1] != 3:
            raise ValueError(f"Expected key_points_optimized shaped (N,3); got {_xyz.shape}")

        if _ids.shape[0] != _xyz.shape[0]:
            _min = int(min(_ids.shape[0], _xyz.shape[0]))
            _xyz = _xyz[:_min]
            _ids = _ids[:_min]

        _m = np.isfinite(_xyz).all(axis=1)
        _xyz = _xyz[_m]
        _ids = _ids[_m]

        # De-dup by (obj_id, kf_idx)
        global_landmarks_updates[:] = [
            u
            for u in global_landmarks_updates
            if (int(u.get("obj_id", -1)), int(u.get("kf_idx", -1))) != (_obj_id, _kf_idx)
        ]

        # Also store pose/GT + measurement arrays for landmark error analysis
        _pose_est = np.asarray(opt_result.pose_optimized, dtype=float).copy()
        _pose_gt = target_fd.get("gt_pose", None)
        if _pose_gt is not None:
            _pose_gt = np.asarray(_pose_gt, dtype=float).copy()

        _meas_xyz_cam = np.asarray(getattr(target_object_frame_data, "reg_cur_3d", np.zeros((0, 3))), dtype=float)
        if _meas_xyz_cam.ndim == 1 and _meas_xyz_cam.size == 0:
            _meas_xyz_cam = np.zeros((0, 3), dtype=float)
        if _meas_xyz_cam.ndim == 2 and _meas_xyz_cam.shape[1] != 3:
            _meas_xyz_cam = _meas_xyz_cam.reshape(-1, 3)

        _meas_ids = np.asarray(getattr(target_object_frame_data, "reg_valid_idx", np.zeros((0,), dtype=int)), dtype=int).reshape(-1)
        _meas_inliers = np.asarray(getattr(target_object_frame_data, "reg_inliers", np.ones((_meas_ids.shape[0],), dtype=bool)), dtype=bool).reshape(-1)

        _min = int(min(_meas_xyz_cam.shape[0], _meas_ids.shape[0], _meas_inliers.shape[0]))
        _meas_xyz_cam = _meas_xyz_cam[:_min]
        _meas_ids = _meas_ids[:_min]
        _meas_inliers = _meas_inliers[:_min]

        global_landmarks_updates.append(
            {
                "obj_id": _obj_id,
                "kf_idx": _kf_idx,
                "frame_id": _frame_id,
                "xyz": _xyz,
                "ids": _ids,
                "pose_est": _pose_est,
                "pose_gt": _pose_gt,
                "meas_xyz_cam": _meas_xyz_cam,
                "meas_ids": _meas_ids,
                "meas_inliers": _meas_inliers,
            }
        )
    except Exception as _e:
        print(f"Warning: failed to snapshot target keyframe landmarks: {_e}")

if len(iteration_errors) > 0:
    # Check if this was the first frame (no optimization)
    first_frame_note = iteration_tracker.iteration_data[0].get('note', '')
    if first_frame_note:
        print(f"\n{first_frame_note}")
        print(f"Frame added to graph with prior factor. Graph size: {tracking_opt._graph.size() if hasattr(tracking_opt, '_graph') else 'N/A'}")
        print("Note: First frame in graph does not get optimized - it only establishes the reference frame.")
    else:
        print(f"\nOptimization completed. {len(iteration_errors)} iterations tracked.")
        print(f"Initial error: {iteration_errors[0]:.6f}")
        print(f"Final error: {iteration_errors[-1]:.6f}")
        if iteration_errors[0] > 0:
            print(f"Error reduction: {iteration_errors[0] - iteration_errors[-1]:.6f} ({100*(iteration_errors[0] - iteration_errors[-1])/iteration_errors[0]:.2f}%)")
        else:
            print("Initial error was zero - no reduction possible")
    
    # Update trajectories (keep ALL streams aligned at the target frame)
    # Some later plotting cells assume the same length for every traj_* list.
    for _name in [
        'traj_frontend',
        'traj_local',
        'traj_global',
        'traj_gt',
        'traj_local_logged',
        'traj_obj_pose_logged',
    ]:
        if _name in globals() and not isinstance(globals()[_name], list):
            globals()[_name] = list(globals()[_name])

    _pose_frontend = target_fd['pose_frontend'].copy()
    _pose_local_logged = (
        target_fd['pose_local'].copy() if target_fd.get('pose_local', None) is not None else None
    )
    _pose_obj_logged = (
        target_fd['obj_pose_logged'].copy()
        if target_fd.get('obj_pose_logged', None) is not None
        else None
    )
    _pose_rerun = (
        opt_result.pose_optimized.copy()
        if (opt_result is not None and opt_result.pose_optimized is not None)
        else _pose_frontend.copy()
    )
    _pose_gt = target_fd['gt_pose'].copy() if target_fd['gt_pose'] is not None else None

    # Append ONLY if this stream hasn't reached the target frame yet (prevents duplicates on re-run)
    if len(traj_frontend) == frame_idx:
        traj_frontend.append(_pose_frontend)
    if len(traj_local_logged) == frame_idx:
        traj_local_logged.append(_pose_local_logged)
    if len(traj_obj_pose_logged) == frame_idx:
        traj_obj_pose_logged.append(_pose_obj_logged)
    if len(traj_local) == frame_idx:
        traj_local.append(_pose_rerun)
    if len(traj_global) == frame_idx:
        traj_global.append(_pose_rerun)
    if len(traj_gt) == frame_idx:
        traj_gt.append(_pose_gt)
else:
    print("\nWarning: No iteration data was collected. The frame may have already been in the graph.")

    # Still add to trajectories (keep ALL streams aligned at the target frame)
    for _name in [
        'traj_frontend',
        'traj_local',
        'traj_global',
        'traj_gt',
        'traj_local_logged',
        'traj_obj_pose_logged',
    ]:
        if _name in globals() and not isinstance(globals()[_name], list):
            globals()[_name] = list(globals()[_name])

    _pose_frontend = target_fd['pose_frontend'].copy()
    _pose_local_logged = (
        target_fd['pose_local'].copy() if target_fd.get('pose_local', None) is not None else None
    )
    _pose_obj_logged = (
        target_fd['obj_pose_logged'].copy()
        if target_fd.get('obj_pose_logged', None) is not None
        else None
    )
    _pose_rerun = _pose_frontend.copy()
    _pose_gt = target_fd['gt_pose'].copy() if target_fd['gt_pose'] is not None else None

    # Append ONLY if this stream hasn't reached the target frame yet (prevents duplicates on re-run)
    if len(traj_frontend) == frame_idx:
        traj_frontend.append(_pose_frontend)
    if len(traj_local_logged) == frame_idx:
        traj_local_logged.append(_pose_local_logged)
    if len(traj_obj_pose_logged) == frame_idx:
        traj_obj_pose_logged.append(_pose_obj_logged)
    if len(traj_local) == frame_idx:
        traj_local.append(_pose_rerun)
    if len(traj_global) == frame_idx:
        traj_global.append(_pose_rerun)
    if len(traj_gt) == frame_idx:
        traj_gt.append(_pose_gt)


## Visualizations

### Per-Iteration Optimization Progress


In [ ]:
# Visualize per-iteration optimization progress
if len(iteration_errors) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Error over iterations
    ax = axes[0, 0]
    iterations = [d['iteration'] for d in iteration_tracker.iteration_data]
    ax.plot(iterations, iteration_errors, 'b-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Error')
    ax.set_title(f'Optimization Error vs Iteration (Frame {FRAME_NUMBER})')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

# Error change per iteration
ax = axes[0, 1]
if len(iteration_changes) > 1:
    ax.plot(iterations[1:], iteration_changes[1:], 'g-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Error Change')
    ax.set_title('Error Reduction per Iteration')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)

# Relative error change
ax = axes[1, 0]
if len(iteration_errors) > 1:
    relative_changes = []
    for i in range(1, len(iteration_errors)):
        if iteration_errors[i-1] > 0:
            rel_change = (iteration_errors[i-1] - iteration_errors[i]) / iteration_errors[i-1] * 100
            relative_changes.append(rel_change)
        else:
            relative_changes.append(0)
    ax.plot(iterations[1:], relative_changes, 'r-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Relative Error Reduction (%)')
    ax.set_title('Relative Error Reduction per Iteration')
    ax.grid(True, alpha=0.3)

# Cumulative error reduction
ax = axes[1, 1]
if len(iteration_errors) > 0:
    initial_error = iteration_errors[0]
    cumulative_reduction = [(initial_error - err) / initial_error * 100 for err in iteration_errors]
    ax.plot(iterations, cumulative_reduction, 'm-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cumulative Error Reduction (%)')
    ax.set_title('Cumulative Error Reduction')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No iteration data available for visualization.")
    print("This may happen if:")
    print("  - The frame was already optimized in the graph")
    print("  - The optimization was skipped (e.g., first frame)")
    print("  - An error occurred during optimization tracking")


### Per-Iteration Pose Changes


In [ ]:
# Extract pose changes per iteration
if any(p is not None for p in iteration_poses):
    valid_poses = [p for p in iteration_poses if p is not None]
    valid_iterations = [i for i, p in enumerate(iteration_poses) if p is not None]
    
    if len(valid_poses) > 1:
        # Extract translation and rotation components
        translations = np.array([p[:3, 3] for p in valid_poses])
        rotations = np.array([R.from_matrix(p[:3, :3]).as_euler('xyz', degrees=True) for p in valid_poses])
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        # Translation components
        for i, (ax, label, color) in enumerate(zip(axes[0], ['X', 'Y', 'Z'], ['r', 'g', 'b'])):
            ax.plot(valid_iterations, translations[:, i], f'{color}-o', linewidth=2, markersize=6)
            ax.set_xlabel('Iteration')
            ax.set_ylabel(f'Translation {label} (m)')
            ax.set_title(f'Translation {label} per Iteration')
            ax.grid(True, alpha=0.3)
        
        # Rotation components
        for i, (ax, label, color) in enumerate(zip(axes[1], ['Roll', 'Pitch', 'Yaw'], ['r', 'g', 'b'])):
            ax.plot(valid_iterations, rotations[:, i], f'{color}-o', linewidth=2, markersize=6)
            ax.set_xlabel('Iteration')
            ax.set_ylabel(f'Rotation {label} (deg)')
            ax.set_title(f'Rotation {label} per Iteration')
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Compute pose differences between iterations
        pose_diffs_trans = []
        pose_diffs_rot = []
        for i in range(1, len(valid_poses)):
            # Translation difference
            trans_diff = np.linalg.norm(translations[i] - translations[i-1])
            pose_diffs_trans.append(trans_diff)
            
            # Rotation difference
            R1 = valid_poses[i-1][:3, :3]
            R2 = valid_poses[i][:3, :3]
            R_diff = R1.T @ R2
            tr = np.trace(R_diff)
            theta = np.arccos(np.clip((tr - 1)/2, -1, 1))
            pose_diffs_rot.append(np.degrees(theta))
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        ax = axes[0]
        ax.plot(valid_iterations[1:], pose_diffs_trans, 'b-o', linewidth=2, markersize=6)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Translation Change (m)')
        ax.set_title('Translation Change per Iteration')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        ax = axes[1]
        ax.plot(valid_iterations[1:], pose_diffs_rot, 'r-o', linewidth=2, markersize=6)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Rotation Change (deg)')
        ax.set_title('Rotation Change per Iteration')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No pose data available for per-iteration tracking.")


In [ ]:
# --- Per-iteration pose error vs GT ---
# Computes translation/rotation error of the target-frame pose estimate at each LM iteration.
# GT is normalized to the first valid GT frame via: GT_rel = GT @ inv(GT0) (removes the global offset).

if len(iteration_errors) == 0:
    print("No iteration data available for GT error plot.")
elif target_fd.get("gt_pose", None) is None:
    print(
        "No dataset GT for this frame. Load GT (Cell 8), re-run the replay cell, then re-run the per-iteration tracking cell."
    )
else:
    # Collect per-iteration pose matrices (as stored by IterationTrackingOptimizer)
    iters = []
    T_iter_raw = []
    for d in iteration_tracker.iteration_data:
        T = d.get("values", None)
        if T is None:
            continue
        it = d.get("iteration", None)
        iters.append(int(it) if it is not None else len(iters))
        T_iter_raw.append(np.asarray(T, dtype=float))

    if len(T_iter_raw) == 0:
        print("No per-iteration pose matrices found in iteration_tracker.iteration_data.")
    else:
        def _pose_err(T_pred: np.ndarray, T_ref: np.ndarray):
            t_err = float(np.linalg.norm(T_pred[:3, 3] - T_ref[:3, 3]))
            R = T_pred[:3, :3].T @ T_ref[:3, :3]
            tr = float(np.trace(R))
            theta = float(np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0)))
            r_err_deg = float(np.degrees(theta))
            return t_err, r_err_deg

        # Decide pose convention: IterationTrackingOptimizer stores Pose3.matrix() (c2w in our pipeline).
        # opt_result.pose_optimized is inverse_SE3(Pose3.matrix()). We'll auto-detect using the last iteration.
        use_inverse = True
        if (
            "opt_result" in globals()
            and opt_result is not None
            and getattr(opt_result, "pose_optimized", None) is not None
        ):
            T_opt = np.asarray(opt_result.pose_optimized, dtype=float)
            t0, r0 = _pose_err(T_iter_raw[-1], T_opt)
            t1, r1 = _pose_err(inverse_SE3(T_iter_raw[-1]), T_opt)
            # Weighted score (meters + small weight on degrees)
            use_inverse = (t1 + 0.01 * r1) < (t0 + 0.01 * r0)

        T_iter = [inverse_SE3(T) if use_inverse else T for T in T_iter_raw]
        print(
            f"Pose convention used for per-iter estimates: {'inverse_SE3(Pose3.matrix())' if use_inverse else 'Pose3.matrix()'}"
        )

        # Normalize GT to the first valid GT frame (if GT is already normalized, this is a no-op)
        T_gt_raw = np.asarray(target_fd["gt_pose"], dtype=float)
        first_gt = None
        if "traj_gt" in globals() and isinstance(traj_gt, list):
            for g in traj_gt:
                if g is not None:
                    first_gt = np.asarray(g, dtype=float)
                    break

        if first_gt is not None:
            T_align = inverse_SE3(first_gt)
            T_gt = T_gt_raw @ T_align
            print("GT normalization: GT @ inv(GT0) (GT0 is first valid GT in traj_gt)")
        else:
            T_gt = T_gt_raw
            print("GT normalization skipped (could not find any valid GT in traj_gt)")

        t_err = []
        r_err = []
        for T in T_iter:
            te, re = _pose_err(T, T_gt)
            t_err.append(te)
            r_err.append(re)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)

        ax = axes[0]
        ax.plot(iters, t_err, "b-o", linewidth=2, markersize=6)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Translation error (m)")
        ax.set_title("Translation error vs GT")
        ax.grid(True, alpha=0.3)
        ax.set_yscale("log")

        ax = axes[1]
        ax.plot(iters, r_err, "r-o", linewidth=2, markersize=6)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Rotation error (deg)")
        ax.set_title("Rotation error vs GT")
        ax.grid(True, alpha=0.3)
        ax.set_yscale("log")

        plt.suptitle(f"Per-iteration pose error vs GT (frame {FRAME_NUMBER})")
        plt.tight_layout()
        plt.show()

In [ ]:
# --- Per-iteration landmark changes (target optimization) ---
# Visualize how the optimized landmark 3D positions move across LM iterations.
# Uses iteration_tracker.iteration_values (gtsam.Values snapshots).

ONLY_INLIERS = True
USE_TARGET_LANDMARK_SET = False  # if False -> plot all global landmarks in the optimizer

if "iteration_tracker" not in globals() or not hasattr(iteration_tracker, "iteration_values"):
    print("No iteration_tracker.iteration_values found. Run the per-iteration tracking cell first.")
elif len(iteration_tracker.iteration_values) == 0:
    print("iteration_tracker.iteration_values is empty (likely first-frame no-optimization).")
else:
    vals_list = iteration_tracker.iteration_values

    # Choose landmark ids to track
    lm_ids = None
    if USE_TARGET_LANDMARK_SET and "target_object_frame_data" in globals() and target_object_frame_data is not None:
        lm_ids = np.asarray(getattr(target_object_frame_data, "reg_valid_idx", []), dtype=int).reshape(-1)
        if ONLY_INLIERS and getattr(target_object_frame_data, "reg_inliers", None) is not None:
            _m = np.asarray(target_object_frame_data.reg_inliers, dtype=bool).reshape(-1)
            if _m.shape[0] == lm_ids.shape[0]:
                lm_ids = lm_ids[_m]
    else:
        if hasattr(iteration_tracker.base_optimizer, "inserted_landmark_ids"):
            lm_ids = np.asarray(iteration_tracker.base_optimizer.inserted_landmark_ids, dtype=int).reshape(-1)

    if lm_ids is None or lm_ids.size == 0:
        print("No landmark ids available to visualize.")
    else:
        lm_ids = np.unique(lm_ids)

        def _extract_landmarks(values: gtsam.Values, ids: np.ndarray) -> np.ndarray:
            xyz = np.full((ids.shape[0], 3), np.nan, dtype=float)
            for j, lid in enumerate(ids.tolist()):
                Lj = gtsam.symbol("l", int(lid))
                if values.exists(Lj):
                    xyz[j, :] = np.asarray(values.atPoint3(Lj), dtype=float).reshape(3,)
            return xyz

        xyz_by_it = []
        for v in vals_list:
            xyz_by_it.append(_extract_landmarks(v, lm_ids))

        # Compute step-wise displacement stats (||p_k - p_{k-1}||)
        mean_step = [0.0]
        max_step = [0.0]
        for k in range(1, len(xyz_by_it)):
            a = xyz_by_it[k - 1]
            b = xyz_by_it[k]
            m = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
            if not np.any(m):
                mean_step.append(float("nan"))
                max_step.append(float("nan"))
            else:
                d = np.linalg.norm(b[m] - a[m], axis=1)
                mean_step.append(float(np.mean(d)))
                max_step.append(float(np.max(d)))

        print(f"Tracked landmarks: {lm_ids.shape[0]}  |  LM iterations captured: {len(xyz_by_it)}")

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Baseline = iteration 0
        xyz0 = xyz_by_it[0]
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()
        

        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color="rgba(140,140,140,0.45)"),
                text=[f"id={int(i)}" for i in lm_ids[m0]],
                hovertemplate="%{text}<extra></extra>",
                name="iter 0 (baseline)",
            )
        )

        # Add one trace per iteration; slider will toggle visibility
        cmin = float(np.min(lm_ids))
        cmax = float(np.max(lm_ids))

        for k, xyz in enumerate(xyz_by_it):
            mk = np.isfinite(xyz).all(axis=1)
            fig.add_trace(
                go.Scatter3d(
                    x=xyz[mk, 0],
                    y=xyz[mk, 1],
                    z=xyz[mk, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=lm_ids[mk].astype(float),
                        colorscale="Turbo",
                        cmin=cmin,
                        cmax=cmax,
                        opacity=0.9,
                        colorbar=dict(title="landmark id"),
                    ),
                    text=[f"id={int(i)}" for i in lm_ids[mk]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"iter {k}",
                    visible=(k == 0),
                )
            )

        steps = []
        for k in range(len(xyz_by_it)):
            # trace 0 is baseline; traces 1.. are iterations
            vis = [True] + [False] * len(xyz_by_it)
            vis[1 + k] = True
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Landmarks over LM iterations (frame {FRAME_NUMBER}) — iter {k}",
                        },
                    ],
                    label=str(k),
                )
            )

        fig.update_layout(
            title=f"Landmarks over LM iterations (frame {FRAME_NUMBER}) — iter 0",
            margin=dict(l=0, r=0, b=0, t=45),
            scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
            sliders=[dict(active=0, currentvalue={"prefix": "iter: "}, steps=steps)],
            legend=dict(x=0.01, y=0.99),
        )

        fig.show()

        # Also plot step-size stats (how much landmarks moved each iteration)
        fig2, ax = plt.subplots(1, 1, figsize=(10, 3))
        ax.plot(range(len(mean_step)), mean_step, "b-o", label="mean ||Δp||")
        ax.plot(range(len(max_step)), max_step, "r-o", label="max ||Δp||")
        ax.set_yscale("log")
        ax.set_xlabel("iteration")
        ax.set_ylabel("meters")
        ax.set_title("Landmark update magnitude per LM iteration")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

### Trajectory Visualization


In [ ]:
# Convert trajectories to numpy arrays
# NOTE on provenance:
# - traj_frontend:       LOGGED meta_data['pose_frontend']
# - traj_local_logged:   LOGGED meta_data['pose_local']
# - traj_obj_pose_logged:LOGGED meta_data['obj_pose'] (pipeline state; can include global updates)
# - traj_local:          RERUN local optimization in this notebook
# - traj_global:         RERUN global optimization in this notebook
traj_frontend = np.asarray(traj_frontend)
traj_local = np.asarray(traj_local)
traj_global = np.asarray(traj_global)
traj_local_logged = np.asarray(traj_local_logged)
traj_obj_pose_logged = np.asarray(traj_obj_pose_logged)

# Logged poses can be missing (None) -> convert to numeric arrays with NaNs
# This keeps plotting code simple (NaNs will break the line where GT/logs are missing).
def _poses_to_nan_array(_poses):
    _poses_arr = np.full((len(_poses), 4, 4), np.nan, dtype=float)
    for _i, _T in enumerate(_poses):
        if _T is None:
            continue
        _T = np.asarray(_T, dtype=float)
        if _T.shape == (4, 4):
            _poses_arr[_i] = _T
    return _poses_arr

traj_local_logged = _poses_to_nan_array(traj_local_logged)
traj_obj_pose_logged = _poses_to_nan_array(traj_obj_pose_logged)

# GT may contain None -> build a numeric array with NaNs
_gt_valid = np.array([g is not None for g in traj_gt], dtype=bool)
traj_gt_arr = np.full((len(traj_gt), 4, 4), np.nan, dtype=float)
for _i, _g in enumerate(traj_gt):
    if _g is not None:
        traj_gt_arr[_i] = np.asarray(_g, dtype=float)

# Align lengths across trajectories (can differ if some cells appended only a subset)
_min_len = min(
    len(traj_frontend),
    len(traj_local_logged),
    len(traj_obj_pose_logged),
    len(traj_local),
    len(traj_global),
    len(traj_gt_arr),
)
traj_frontend = traj_frontend[:_min_len]
traj_local_logged = traj_local_logged[:_min_len]
traj_obj_pose_logged = traj_obj_pose_logged[:_min_len]
traj_local = traj_local[:_min_len]
traj_global = traj_global[:_min_len]
traj_gt_arr = traj_gt_arr[:_min_len]
_gt_valid = _gt_valid[:_min_len]

# 3D Trajectory Plot
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot trajectories
ax.plot(traj_frontend[:, 0, 3], traj_frontend[:, 1, 3], traj_frontend[:, 2, 3],
        'r.-', alpha=0.6, label='Frontend (logged)', linewidth=1.5, markersize=3, markevery=5)
ax.plot(traj_local_logged[:, 0, 3], traj_local_logged[:, 1, 3], traj_local_logged[:, 2, 3],
        'c.-', alpha=0.6, label='Local (logged)', linewidth=1.5, markersize=3, markevery=5)
ax.plot(traj_obj_pose_logged[:, 0, 3], traj_obj_pose_logged[:, 1, 3], traj_obj_pose_logged[:, 2, 3],
        'g.-', alpha=0.6, label='Obj pose (logged)', linewidth=1.5, markersize=3, markevery=5)
ax.plot(traj_local[:, 0, 3], traj_local[:, 1, 3], traj_local[:, 2, 3],
        'b.-', alpha=0.6, label='Local (rerun)', linewidth=1.5, markersize=3, markevery=5)
ax.plot(traj_global[:, 0, 3], traj_global[:, 1, 3], traj_global[:, 2, 3],
        'm.-', alpha=0.5, label='Global (rerun)', linewidth=1.5, markersize=3, markevery=5)

# Highlight keyframes
if len(keyframe_frame_ids) > 0:
    kf_indices = [
        i
        for i, fid in enumerate(meta_data['frame_id'][:frame_idx+1])
        if fid in keyframe_frame_ids and i < _min_len
    ]
    kf_array = np.array(kf_indices)
    if len(kf_array) > 0:
        ax.scatter(traj_global[kf_array, 0, 3], 
                   traj_global[kf_array, 1, 3], 
                   traj_global[kf_array, 2, 3], 
                   c='m', marker='o', s=150, alpha=0.9, label='Global Opt (Keyframes)', 
                   edgecolors='darkmagenta', linewidths=2, zorder=5)

# Highlight target frame
if frame_idx < _min_len:
    ax.scatter(traj_local[frame_idx, 0, 3], 
               traj_local[frame_idx, 1, 3], 
               traj_local[frame_idx, 2, 3], 
               c='orange', marker='*', s=500, alpha=1.0, label=f'Target Frame {FRAME_NUMBER}', 
               edgecolors='red', linewidths=2, zorder=10)
else:
    print(f"Warning: frame_idx={frame_idx} is out of bounds for plotted trajectories (len={_min_len}).")

if np.any(_gt_valid):
    ax.plot(traj_gt_arr[:, 0, 3], traj_gt_arr[:, 1, 3], traj_gt_arr[:, 2, 3],
            'k.-', label='GT (dataset)', linewidth=2, markersize=4, markevery=5, alpha=0.7)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.legend()
ax.set_title(f"3D Trajectory up to Frame {FRAME_NUMBER}")
plt.show()

# Translation components over time
fig, axes = plt.subplots(3, 1, figsize=(15, 10))
frames = np.arange(len(traj_frontend))

for i, (ax, label) in enumerate(zip(axes, ['X', 'Y', 'Z'])):
    ax.plot(frames, traj_frontend[:, i, 3], 'r-', alpha=0.7, label='Frontend (logged)', linewidth=2)
    ax.plot(frames, traj_local_logged[:, i, 3], 'c-', alpha=0.7, label='Local (logged)', linewidth=2)
    ax.plot(frames, traj_obj_pose_logged[:, i, 3], 'g-', alpha=0.7, label='Obj pose (logged)', linewidth=2)
    ax.plot(frames, traj_local[:, i, 3], 'b-', label='Local (rerun)', linewidth=2)
    ax.plot(frames, traj_global[:, i, 3], 'm-', alpha=0.5, label='Global (rerun)', linewidth=2)
    
    # Highlight keyframes
    if len(keyframe_frame_ids) > 0:
        kf_indices = [
            j
            for j, fid in enumerate(meta_data['frame_id'][:frame_idx+1])
            if fid in keyframe_frame_ids and j < len(frames)
        ]
        if len(kf_indices) > 0:
            ax.scatter(
                [frames[j] for j in kf_indices],
                [traj_global[j, i, 3] for j in kf_indices],
                c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                edgecolors='darkmagenta', linewidths=1.5, zorder=5,
            )

    # Highlight target frame
    if frame_idx < len(frames):
        ax.scatter(
            frames[frame_idx], traj_local[frame_idx, i, 3],
            c='orange', marker='*', s=300, alpha=1.0, label=f'Target Frame {FRAME_NUMBER}',
            edgecolors='red', linewidths=2, zorder=10,
        )
    
    if np.any(_gt_valid):
        ax.plot(frames, traj_gt_arr[:, i, 3], 'k-', label='GT (dataset)', linewidth=2, alpha=0.7)
    
    ax.set_title(f"Translation {label} over Time")
    ax.set_xlabel("Frame")
    ax.set_ylabel(f"{label} (m)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Error Analysis


In [ ]:
# Error analysis
# IMPORTANT:
# - `meta_data['obj_pose']` is the pipeline's estimated pose at that frame (NOT ground truth).
# - True GT is not stored in meta_data; you must load it from the dataset (see GT-loading cell).


def rot_err_deg(T_pred: np.ndarray, T_gt: np.ndarray) -> float:
    R1 = T_pred[:3, :3]
    R2 = T_gt[:3, :3]
    R = R1.T @ R2
    tr = float(np.trace(R))
    theta = np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0))
    return float(np.degrees(theta))


def align_pred_to_gt(pred: np.ndarray, gt: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    """Align pred to GT by matching the first valid frame (same as experiments/ho3d/evaluate_ho3d_single.py)."""
    first = int(np.where(valid_mask)[0][0])
    return pred @ inverse_SE3(pred[first]) @ gt[first]


def compute_pose_errors(pred: np.ndarray, ref: np.ndarray, valid_mask: np.ndarray):
    idxs = np.where(valid_mask)[0]
    t_err = np.linalg.norm(pred[idxs, :3, 3] - ref[idxs, :3, 3], axis=1)
    r_err = np.array([rot_err_deg(pred[i], ref[i]) for i in idxs], dtype=float)
    return idxs, t_err, r_err


# Arrays (some are logged, some are rerun)
traj_frontend_arr = np.asarray(traj_frontend)              # logged
traj_local_logged_arr = np.asarray(traj_local_logged)      # logged
traj_obj_logged_arr = np.asarray(traj_obj_pose_logged)     # logged (pipeline output pose)
traj_local_rerun_arr = np.asarray(traj_local)              # rerun
traj_global_rerun_arr = np.asarray(traj_global)            # rerun

N = traj_frontend_arr.shape[0]
frame_ids_arr = np.asarray(meta_data['frame_id'][:N], dtype=int)

# Build GT array (may be all-None)
gt_valid = np.array([g is not None for g in traj_gt], dtype=bool)
traj_gt_full = np.full((len(traj_gt), 4, 4), np.nan, dtype=float)
for i, g in enumerate(traj_gt):
    if g is not None:
        traj_gt_full[i] = np.asarray(g, dtype=float)

print("\n=== Error Analysis Provenance ===")
print("Frontend (logged):        meta_data['pose_frontend']")
print("Local (logged):           meta_data['pose_local']")
print("Obj pose (logged):        meta_data['obj_pose'] (pipeline state, NOT GT)")
print("Local (rerun):            LocalOptimizer.optimize(...) in notebook")
print("Global (rerun):           KeyFrameGraph.update(...) in notebook")
print("GT (dataset):             loaded separately (None if not loaded)")

# ---------------------------------------------------------------------
# 1) Error vs LOGGED obj_pose (consistency check, NOT GT)
# ---------------------------------------------------------------------
ref_valid = np.isfinite(traj_obj_logged_arr[:, 0, 0])
if np.any(ref_valid):
    ref = traj_obj_logged_arr
    preds = {
        'Frontend (logged)': traj_frontend_arr,
        'Local (logged)': traj_local_logged_arr,
        'Local (rerun)': traj_local_rerun_arr,
        'Global (rerun)': traj_global_rerun_arr,
    }

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax_t, ax_r = axes

    for name, P in preds.items():
        idxs, t_err, r_err = compute_pose_errors(P, ref, ref_valid)
        ax_t.plot(frame_ids_arr[idxs], t_err, lw=2, label=name)
        ax_r.plot(frame_ids_arr[idxs], r_err, lw=2, label=name)

    ax_t.set_title("Pose error vs LOGGED obj_pose (pipeline state; NOT ground truth)")
    ax_t.set_ylabel("translation error")
    ax_t.grid(True, alpha=0.3)
    ax_t.legend()

    ax_r.set_ylabel("rotation error (deg)")
    ax_r.set_xlabel("frame_id")
    ax_r.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------
# 2) Error vs DATASET GT (only if GT loaded)
# ---------------------------------------------------------------------
if not np.any(gt_valid):
    print("\nNo dataset GT loaded. To enable GT, load gt_pose_by_frame_id before running the replay cell.")
    print("(After loading GT, re-run the replay cell so traj_gt is populated.)")
else:
    gt = traj_gt_full

    # Align each predicted trajectory to GT using the first valid GT frame
    preds = {
        'Frontend (logged)': traj_frontend_arr,
        'Local (logged)': traj_local_logged_arr,
        'Obj pose (logged)': traj_obj_logged_arr,
        'Local (rerun)': traj_local_rerun_arr,
        'Global (rerun)': traj_global_rerun_arr,
    }

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax_t, ax_r = axes

    for name, P in preds.items():
        P_aligned = align_pred_to_gt(P, gt, gt_valid)
        idxs, t_err, r_err = compute_pose_errors(P_aligned, gt, gt_valid)
        ax_t.plot(frame_ids_arr[idxs], t_err, lw=2, label=name)
        ax_r.plot(frame_ids_arr[idxs], r_err, lw=2, label=name)

    ax_t.set_title("Pose error vs DATASET GT (pred aligned to GT by first valid frame)")
    ax_t.set_ylabel("translation error")
    ax_t.grid(True, alpha=0.3)
    ax_t.legend()

    ax_r.set_ylabel("rotation error (deg)")
    ax_r.set_xlabel("frame_id")
    ax_r.grid(True, alpha=0.3)

    # Highlight target frame if present and GT exists there
    if 0 <= frame_idx < len(frame_ids_arr) and gt_valid[frame_idx]:
        xf = int(frame_ids_arr[frame_idx])
        ax_t.axvline(x=xf, color='k', linestyle='--', alpha=0.3)
        ax_r.axvline(x=xf, color='k', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print target-frame numbers (vs GT) if available
    if 0 <= frame_idx < len(frame_ids_arr) and gt_valid[frame_idx]:
        print("\n=== Target-frame error vs GT (aligned) ===")
        xf = int(frame_ids_arr[frame_idx])
        print(f"Target frame_id={xf} (meta idx={frame_idx})")
        for name, P in preds.items():
            P_aligned = align_pred_to_gt(P, gt, gt_valid)
            t_err = float(np.linalg.norm(P_aligned[frame_idx, :3, 3] - gt[frame_idx, :3, 3]))
            r_err = float(rot_err_deg(P_aligned[frame_idx], gt[frame_idx]))
            print(f"  {name:18s}  t_err={t_err:.6f}  r_err={r_err:.2f} deg")
    else:
        print("\nTarget frame has no GT entry (or frame_idx out of range).")


### Frame Data Summary


In [ ]:
# Print summary information about the target frame
print("=" * 60)
print(f"FRAME {FRAME_NUMBER} SUMMARY")
print("=" * 60)
print(f"\nFrame Index: {frame_idx}")
print(f"Is Keyframe: {is_target_keyframe}")
print(f"\nRegistration Data:")
print(f"  Number of 3D points: {len(target_fd['cur_3d'])}")
print(f"  Number of inliers: {np.sum(target_fd['inliers'])}")
print(f"  Mean residual: {np.mean(target_fd['residuals']):.6f}")
print(f"  Max residual: {np.max(target_fd['residuals']) if len(target_fd['residuals']) > 0 else 0:.6f}")
print(f"\nPoses (provenance):")
print(f"  Frontend pose translation (logged): {target_fd['pose_frontend'][:3, 3]}")
print(f"  Local pose translation (logged):    {target_fd['pose_local'][:3, 3]}")
if target_fd.get('obj_pose_logged', None) is not None:
    print(f"  Obj pose translation (logged):      {target_fd['obj_pose_logged'][:3, 3]}")

# This is the pose produced by the notebook's per-iteration rerun (if you ran it)
if opt_result is not None:
    print(f"  Optimized pose translation (rerun): {opt_result.pose_optimized[:3, 3]}")

# True GT only exists if you loaded it from the dataset
if target_fd['gt_pose'] is not None:
    print(f"  GT pose translation (dataset):      {target_fd['gt_pose'][:3, 3]}")
else:
    print("  GT pose:                            (not loaded)")

print(f"\nOptimization Iterations: {len(iteration_errors)}")
if len(iteration_errors) > 0:
    print(f"  Initial error: {iteration_errors[0]:.6f}")
    print(f"  Final error: {iteration_errors[-1]:.6f}")
    if iteration_errors[0] > 0:
        print(f"  Total reduction: {iteration_errors[0] - iteration_errors[-1]:.6f} ({100*(iteration_errors[0] - iteration_errors[-1])/iteration_errors[0]:.2f}%)")
    else:
        print("  Initial error was zero - no reduction possible")
else:
    print("  No iteration data collected.")
    print("  Possible reasons:")
    print("    - Frame was already in the graph")
    print("    - Optimization was skipped (e.g., first frame)")
    print("    - Error during tracking")


## Graph / Factor Diagnostics (Pose ↔ Landmark Edges)

This section introspects the *actual* GTSAM factor graph inside the global optimizer and visualizes:
- Factor types + counts
- Pose↔landmark edges and their error contributions
- Per-keyframe and per-landmark degree/error statistics


In [ ]:
from collections import Counter, defaultdict


def _sym_chr_idx(k: int):
    s = gtsam.Symbol(k)
    c = s.chr()
    # gtsam Python may return either a 1-char string or an int ASCII code
    if isinstance(c, (int, np.integer)):
        c = chr(int(c))
    return c, int(s.index())


def _rot_err_deg(T1: np.ndarray, T2: np.ndarray) -> float:
    R1 = T1[:3, :3]
    R2 = T2[:3, :3]
    R = R1.T @ R2
    tr = float(np.trace(R))
    theta = np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0))
    return float(np.degrees(theta))


def extract_factor_records(graph: gtsam.NonlinearFactorGraph, values: gtsam.Values):
    recs = []
    for fi in range(graph.size()):
        f = graph.at(fi)
        try:
            keys = list(f.keys())
        except Exception:
            keys = []
        try:
            err = float(f.error(values))
        except Exception:
            err = float('nan')
        recs.append({
            'factor_index': fi,
            'type': type(f).__name__,
            'keys': keys,
            'error': err,
        })
    return recs


def extract_pose_landmark_edges(
    graph: gtsam.NonlinearFactorGraph,
    values: gtsam.Values,
    pose_chr: str = 'x',
    lm_chr: str = 'l',
):
    edges = []
    for fi in range(graph.size()):
        f = graph.at(fi)
        try:
            keys = list(f.keys())
        except Exception:
            continue
        if len(keys) != 2:
            continue

        try:
            c0, i0 = _sym_chr_idx(keys[0])
            c1, i1 = _sym_chr_idx(keys[1])
        except Exception:
            continue

        if {c0, c1} != {pose_chr, lm_chr}:
            continue

        pose_idx = i0 if c0 == pose_chr else i1
        lm_id = i0 if c0 == lm_chr else i1

        try:
            err = float(f.error(values))
        except Exception:
            err = float('nan')

        edges.append({
            'factor_index': fi,
            'pose_idx': int(pose_idx),
            'landmark_id': int(lm_id),
            'error': err,
            'type': type(f).__name__,
        })
    return edges


# --- Pull the live global graph from the pipeline replay ---
global_opt = kf_graph._get_optimizer(0)
if not hasattr(global_opt, '_graph') or not hasattr(global_opt, '_values'):
    raise RuntimeError("Global optimizer does not expose _graph/_values. Expected LMGraphOptimizer.")

graph = global_opt._graph
values = global_opt._values

print("=== Global graph summary ===")
print(f"Factors: {graph.size()}")
print(f"Values:  {values.size()}")

factor_recs = extract_factor_records(graph, values)
edges = extract_pose_landmark_edges(graph, values)

print(f"Pose↔landmark edges (2-key x-l factors): {len(edges)}")

# Factor type counts
ft_counts = Counter([r['type'] for r in factor_recs])

# Edge stats
edge_err = np.array([e['error'] for e in edges], dtype=float)
edge_err = edge_err[np.isfinite(edge_err)]

pose_degree = defaultdict(int)
pose_err_sum = defaultdict(float)
lm_degree = defaultdict(int)
lm_err_sum = defaultdict(float)

for e in edges:
    if not np.isfinite(e['error']):
        continue
    pose_degree[e['pose_idx']] += 1
    pose_err_sum[e['pose_idx']] += float(e['error'])
    lm_degree[e['landmark_id']] += 1
    lm_err_sum[e['landmark_id']] += float(e['error'])

pose_idxs = np.array(sorted(pose_degree.keys()), dtype=int)
# Map keyframe index -> original frame_id for plotting
pose_frame_ids = np.asarray([kf_frame_id_by_kf_idx.get(i, i) for i in pose_idxs], dtype=int)
pose_deg = np.array([pose_degree[i] for i in pose_idxs], dtype=int)
pose_err = np.array([pose_err_sum[i] for i in pose_idxs], dtype=float)
pose_mean_err = pose_err / np.maximum(pose_deg, 1)

lm_ids = np.array(sorted(lm_degree.keys()), dtype=int)
lm_deg = np.array([lm_degree[i] for i in lm_ids], dtype=int)
lm_err = np.array([lm_err_sum[i] for i in lm_ids], dtype=float)
lm_mean_err = lm_err / np.maximum(lm_deg, 1)

# --- Plots ---
fig = plt.figure(figsize=(16, 10))

# 1) Factor type counts
ax = fig.add_subplot(2, 2, 1)
ft_items = sorted(ft_counts.items(), key=lambda kv: kv[1], reverse=True)
ft_names = [k for k, _ in ft_items]
ft_vals = [v for _, v in ft_items]
ax.bar(range(len(ft_names)), ft_vals)
ax.set_xticks(range(len(ft_names)))
ax.set_xticklabels(ft_names, rotation=45, ha='right')
ax.set_title('Factor types in global graph')
ax.set_ylabel('count')
ax.grid(True, axis='y', alpha=0.3)

# 2) Edge error distribution
ax = fig.add_subplot(2, 2, 2)
if edge_err.size > 0:
    # Avoid log(0) by clipping
    clipped = np.clip(edge_err, 1e-12, None)
    ax.hist(clipped, bins=60)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title('Pose↔landmark edge error distribution')
    ax.set_xlabel('factor error (log)')
    ax.set_ylabel('count (log)')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No x-l edges found', ha='center', va='center')
    ax.axis('off')

# 3) Per-keyframe degree & error
ax = fig.add_subplot(2, 2, 3)
if pose_idxs.size > 0:
    ax.plot(pose_frame_ids, pose_err, 'r-', lw=2, label='sum edge error')
    ax.set_xlabel('frame_id')
    ax.set_ylabel('sum edge error', color='r')
    ax.tick_params(axis='y', labelcolor='r')
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.bar(pose_frame_ids, pose_deg, alpha=0.25, color='k', label='degree (# edges)')
    ax2.set_ylabel('degree (# edges)', color='k')
    ax2.tick_params(axis='y', labelcolor='k')
    ax.set_title('Per-keyframe connectivity vs error')
else:
    ax.text(0.5, 0.5, 'No per-keyframe stats', ha='center', va='center')
    ax.axis('off')

# 4) Landmark degree vs error (scatter)
ax = fig.add_subplot(2, 2, 4)
if lm_ids.size > 0:
    ax.scatter(lm_deg, lm_err, s=8, alpha=0.5)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('landmark observations (degree)')
    ax.set_ylabel('sum edge error')
    ax.set_title('Landmark degree vs error')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No landmark stats', ha='center', va='center')
    ax.axis('off')

plt.tight_layout()
plt.show()

# --- Print top outliers ---
if edge_err.size > 0:
    top_edges = sorted([e for e in edges if np.isfinite(e['error'])], key=lambda e: e['error'], reverse=True)[:10]
    print("\nTop-10 edges by factor.error(values):")
    for e in top_edges:
        fid = kf_frame_id_by_kf_idx.get(e['pose_idx'], None)
        print(f"  x_{e['pose_idx']} (frame_id={fid})  <->  l_{e['landmark_id']}    err={e['error']:.6e}   (factor_idx={e['factor_index']})")

if lm_ids.size > 0:
    lm_items = sorted(lm_err_sum.items(), key=lambda kv: kv[1], reverse=True)[:10]
    print("\nTop-10 landmarks by summed edge error:")
    for lid, s in lm_items:
        print(f"  l_{lid}: sum_err={s:.6e}, degree={lm_degree[lid]}")

if pose_idxs.size > 0:
    pose_items = sorted(pose_err_sum.items(), key=lambda kv: kv[1], reverse=True)[:10]
    print("\nTop-10 keyframes by summed edge error:")
    for kidx, s in pose_items:
        fid = kf_frame_id_by_kf_idx.get(kidx, None)
        print(f"  x_{kidx} (frame_id={fid}): sum_err={s:.6e}, degree={pose_degree[kidx]}")


## Symbol & Indexing Sanity Checks

This cell audits **pose** (`x*`) and **landmark** (`l*`) symbols in the live GTSAM `values`, and compares them against:
- keyframe indices (`kf_idx`) created in this notebook
- observed tracker IDs (`kp_track_indices` / `obs_track_indices`) seen so far

It helps answer: **are my symbol indices aligned with keyframe indices and track IDs?**


In [ ]:
from collections import defaultdict
import pandas as pd

def extract_symbol_table(values: gtsam.Values):
    keys = list(values.keys())
    rows = []
    for k in keys:
        try:
            c, idx = _sym_chr_idx(k)
        except Exception:
            continue
        rows.append((c, idx, int(k)))
    df = pd.DataFrame(rows, columns=['sym', 'idx', 'raw_key'])
    return df

sym_df = extract_symbol_table(values)
display(sym_df.groupby('sym')['idx'].agg(['count','min','max']).sort_index())

# Pose key sanity vs keyframes built in this notebook
pose_sym = 'x'
lm_sym = 'l'
pose_idxs_in_values = set(sym_df[sym_df['sym']==pose_sym]['idx'].tolist())
lm_idxs_in_values = set(sym_df[sym_df['sym']==lm_sym]['idx'].tolist())

kf_idxs_expected = set(kf_frame_id_by_kf_idx.keys())
missing_pose = sorted(list(kf_idxs_expected - pose_idxs_in_values))
extra_pose = sorted(list(pose_idxs_in_values - kf_idxs_expected))
print(f"\nPose symbols present: {len(pose_idxs_in_values)} | keyframe indices expected: {len(kf_idxs_expected)}")
print(f"Missing pose symbols for these kf_idx: {missing_pose[:20]}" + (" ..." if len(missing_pose)>20 else ""))
print(f"Extra pose symbols not in kf_idx set: {extra_pose[:20]}" + (" ..." if len(extra_pose)>20 else ""))

# Track IDs seen in all keyframes (what you likely *intend* landmarks to correspond to)
track_ids_seen = set()
# Better: derive track ids from meta_data keyframes before target frame
is_kf = np.asarray(meta_data['is_key_frame'], dtype=bool)
for i in range(frame_idx):
    if not bool(is_kf[i]):
        continue
    fd = get_frame_data(meta_data, i)
    # cur_3d_idx are global track IDs used for that keyframe's observations
    for tid in np.asarray(fd['cur_3d_idx'], dtype=int).reshape(-1,):
        track_ids_seen.add(int(tid))

track_ids_seen = set([t for t in track_ids_seen if t >= 0])

print(f"\nUnique track IDs seen in keyframes before target: {len(track_ids_seen)}")
print(f"Unique landmark IDs in values (l*): {len(lm_idxs_in_values)}")

missing_lm = sorted(list(track_ids_seen - lm_idxs_in_values))
extra_lm = sorted(list(lm_idxs_in_values - track_ids_seen))
print(f"Track IDs that have no corresponding landmark symbol: {missing_lm[:20]}" + (" ..." if len(missing_lm)>20 else ""))
print(f"Landmark IDs that were never observed as track IDs: {extra_lm[:20]}" + (" ..." if len(extra_lm)>20 else ""))

# If counts match but IDs differ, try a *first-seen* alignment heuristic to infer a mapping.
if len(track_ids_seen) == len(lm_idxs_in_values) and (missing_lm or extra_lm):
    print("\nCounts match but IDs differ. Attempting a first-seen alignment (heuristic) ...")
    # First seen keyframe index for each track ID
    first_seen_kf = {}
    for i in range(frame_idx):
        if not bool(is_kf[i]):
            continue
        fd = get_frame_data(meta_data, i)
        # approximate keyframe idx by count of keyframes up to i
        kf_idx = int(np.sum(is_kf[:i]))
        for tid in np.asarray(fd['cur_3d_idx'], dtype=int).reshape(-1,):
            tid = int(tid)
            if tid < 0: 
                continue
            if tid not in first_seen_kf:
                first_seen_kf[tid] = kf_idx
    # First incident pose for each landmark in the graph
    first_inc_pose = defaultdict(lambda: 1e9)
    for e in edges:
        lm = int(e['landmark_id'])
        p = int(e['pose_idx'])
        first_inc_pose[lm] = min(first_inc_pose[lm], p)
    # Sort and pair
    tr_sorted = sorted(track_ids_seen, key=lambda t: (first_seen_kf.get(t, 1e9), t))
    lm_sorted = sorted(lm_idxs_in_values, key=lambda l: (first_inc_pose.get(l, 1e9), l))
    mapping = dict(zip(tr_sorted, lm_sorted))
    # Show a small sample
    sample = list(mapping.items())[:20]
    print("Sample track_id -> landmark_id mapping (heuristic):")
    for t,l in sample:
        print(f"  {t} -> {l} (first_seen_kf={first_seen_kf.get(t,None)}, first_inc_pose={first_inc_pose.get(l,None)})")

# Visualization: first-seen comparison (helps spot systematic offsets)
if len(track_ids_seen) > 0 and len(lm_idxs_in_values) > 0:
    # Scatter of sorted IDs for a quick gut check
    fig = plt.figure(figsize=(10,4))
    ax = fig.add_subplot(1,1,1)
    ax.plot(sorted(list(track_ids_seen))[:500], label='track IDs (sorted)')
    ax.plot(sorted(list(lm_idxs_in_values))[:500], label='landmark IDs (sorted)')
    ax.set_title('ID distribution sanity check (first 500 sorted)')
    ax.set_xlabel('rank')
    ax.set_ylabel('id')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


## Per-Factor Chi² / DOF + Noise Model Inspection

`factor.error(values)` returns **0.5 * chi²** (squared Mahalanobis distance). This cell computes:
- `chi2 = 2*error`
- `dof = factor.dim()` when available
- `chi2_per_dof`

Then it shows the **worst factors** and how their errors distribute. This is the fastest way to spot a few bad correspondences or a too-tight covariance.


In [ ]:
def _try_dim(f):
    for attr in ['dim', 'dimension']:
        if hasattr(f, attr):
            try:
                return int(getattr(f, attr)())
            except Exception:
                pass
    return None

def _noise_model_summary(f):
    # Best-effort: not all factor wrappers expose the noise model cleanly in Python
    nm = None
    for attr in ['noiseModel', 'get_noiseModel', 'noise_model']:
        if hasattr(f, attr):
            try:
                nm = getattr(f, attr)()
                break
            except Exception:
                pass
    if nm is None:
        return None
    name = type(nm).__name__
    # Extract sigmas if diagonal
    sigmas = None
    try:
        if hasattr(nm, 'sigmas'):
            sig = np.asarray(nm.sigmas(), dtype=float).reshape(-1,)
            sigmas = sig
    except Exception:
        pass
    return {'noise_type': name, 'sigmas': sigmas}

rows = []
for fi in range(graph.size()):
    f = graph.at(fi)
    try:
        keys = list(f.keys())
    except Exception:
        keys = []
    err = np.nan
    try:
        err = float(f.error(values))
    except Exception:
        pass
    dim = _try_dim(f)
    chi2 = 2.0*err if np.isfinite(err) else np.nan
    chi2_dof = (chi2 / max(dim, 1)) if (dim is not None and np.isfinite(chi2)) else np.nan
    nm = _noise_model_summary(f)
    rows.append({
        'factor_index': fi,
        'type': type(f).__name__,
        'nkeys': len(keys),
        'keys': keys,
        'error': err,
        'chi2': chi2,
        'dim': dim,
        'chi2_per_dof': chi2_dof,
        'noise_type': None if nm is None else nm['noise_type'],
        'noise_sigmas': None if nm is None else nm['sigmas'],
    })

df_f = pd.DataFrame(rows)
display(df_f[['type','nkeys','dim']].value_counts().head(15))

# Worst factors by chi2_per_dof (fallback to chi2 if dim missing)
df_rank = df_f.copy()
df_rank['rank_metric'] = df_rank['chi2_per_dof']
fallback_mask = ~np.isfinite(df_rank['rank_metric'])
df_rank.loc[fallback_mask, 'rank_metric'] = df_rank.loc[fallback_mask, 'chi2']

df_worst = df_rank.sort_values('rank_metric', ascending=False).head(30)
print("\nTop-30 worst factors (rank_metric = chi2/dof if available else chi2):")
display(df_worst[['factor_index','type','dim','chi2','chi2_per_dof','error','keys','noise_type']])

# Histograms
fig = plt.figure(figsize=(14,4))
ax1 = fig.add_subplot(1,2,1)
vals = df_f['chi2'].values
vals = vals[np.isfinite(vals)]
ax1.hist(vals, bins=60)
ax1.set_title('chi² distribution over factors')
ax1.set_xlabel('chi²')
ax1.set_ylabel('count')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(1,2,2)
vals2 = df_f['chi2_per_dof'].values
vals2 = vals2[np.isfinite(vals2)]
if len(vals2) > 0:
    ax2.hist(vals2, bins=60)
    ax2.set_title('chi² / dof distribution (where dof available)')
    ax2.set_xlabel('chi² / dof')
    ax2.set_ylabel('count')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.1, 0.5, 'No factor.dim() available in this GTSAM build', fontsize=12)
plt.show()


## Marginal Covariance Diagnostics

If your uncertainty is wrong, optimization can be dominated by a few constraints.

This cell computes **marginal covariances** for:
- keyframe poses (`x*`) → uses the 6×6 Pose3 covariance; we summarize translation std and rotation std (rough)
- landmarks (`l*`) → 3×3 Point3 covariance; summarize xyz std

Notes:
- `gtsam.Marginals` can be expensive on large graphs. We default to **a subset** (target pose + neighbors + a few worst landmarks).


In [ ]:
def _std_from_cov(cov):
    cov = np.asarray(cov, dtype=float)
    d = np.clip(np.diag(cov), 0.0, np.inf)
    return np.sqrt(d)

def _pose_cov_summaries(margs, pose_idxs):
    out = []
    for i in pose_idxs:
        k = gtsam.symbol('x', int(i))
        try:
            cov = margs.marginalCovariance(k)
        except Exception:
            continue
        std = _std_from_cov(cov)
        # Pose3 is typically [rot(3), trans(3)] in tangent space; but ordering can vary.
        # We report both halves so you can verify the convention.
        out.append({
            'pose_idx': int(i),
            'std0': float(std[0]) if len(std)>0 else np.nan,
            'std1': float(std[1]) if len(std)>1 else np.nan,
            'std2': float(std[2]) if len(std)>2 else np.nan,
            'std3': float(std[3]) if len(std)>3 else np.nan,
            'std4': float(std[4]) if len(std)>4 else np.nan,
            'std5': float(std[5]) if len(std)>5 else np.nan,
        })
    return pd.DataFrame(out)

def _lm_cov_summaries(margs, lm_ids):
    out = []
    for j in lm_ids:
        k = gtsam.symbol('l', int(j))
        try:
            cov = margs.marginalCovariance(k)
        except Exception:
            continue
        std = _std_from_cov(cov)
        out.append({'landmark_id': int(j), 'sx': float(std[0]), 'sy': float(std[1]), 'sz': float(std[2])})
    return pd.DataFrame(out)

# Choose a subset for speed
target_pose_idx = None
if is_target_keyframe:
    # If target is keyframe, its pose idx is the new kf_idx at that moment; approximate using keyframe count up to frame_idx
    target_pose_idx = int(np.sum(np.asarray(meta_data['is_key_frame'][:frame_idx], dtype=bool)))
else:
    # For non-keyframe targets, use the last keyframe before it
    target_pose_idx = int(np.sum(np.asarray(meta_data['is_key_frame'][:frame_idx], dtype=bool)) - 1)
    target_pose_idx = max(target_pose_idx, 0)

# Neighbor poses: those connected to target pose via edges
nbr_pose_idxs = set([target_pose_idx])
for e in edges:
    if int(e['pose_idx']) == int(target_pose_idx):
        nbr_pose_idxs.add(int(e['pose_idx']))
pose_subset = sorted(list(nbr_pose_idxs))

# Landmark subset: worst by mean edge error + those incident to target pose
lm_incident = [int(e['landmark_id']) for e in edges if int(e['pose_idx']) == int(target_pose_idx)]
lm_worst = sorted(lm_err_sum.keys(), key=lambda k: lm_err_sum[k], reverse=True)[:50]
lm_subset = sorted(list(set(lm_incident + [int(x) for x in lm_worst])))[:200]

print(f"Computing marginals for {len(pose_subset)} poses and {len(lm_subset)} landmarks (subset)...")
try:
    margs = gtsam.Marginals(graph, values)
    df_pose_cov = _pose_cov_summaries(margs, pose_subset)
    df_lm_cov = _lm_cov_summaries(margs, lm_subset)
    
    print("\nPose covariance diag std (6D tangent), subset:")
    display(df_pose_cov)
    print("\nLandmark covariance diag std (x,y,z), subset (showing head):")
    display(df_lm_cov.sort_values(['sx','sy','sz'], ascending=False).head(30))

    # Quick plots
    if len(df_lm_cov) > 0:
        fig = plt.figure(figsize=(10,4))
        ax = fig.add_subplot(1,1,1)
        ax.hist(df_lm_cov[['sx','sy','sz']].values.reshape(-1,), bins=60)
        ax.set_title('Landmark marginal std histogram (sx,sy,sz pooled)')
        ax.set_xlabel('std (units of position)')
        ax.set_ylabel('count')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        plt.show()
except Exception as e:
    print("Marginals failed (often due to graph size / numerical issues / missing elimination ordering):")
    print(e)


## Local Observation Outlier Checks

Even if the global optimizer looks wrong, it might be driven by a few bad correspondences coming from the local registration.

This cell uses the **logged per-point residuals/inliers** for the *target frame* to:
- visualize residual distribution
- list the worst track IDs
- check whether those track IDs exist as landmarks in the global graph


In [ ]:
# Target frame residuals from logs
res = np.asarray(target_fd.get('residuals', []), dtype=float).reshape(-1,)
tids = np.asarray(target_fd.get('cur_3d_idx', []), dtype=int).reshape(-1,)
inl = np.asarray(target_fd.get('inliers', []), dtype=bool).reshape(-1,)

if len(res) == 0:
    print('No residuals in target_fd (is your meta_data logging residuals for this stage?)')
else:
    # Robust summary
    finite = np.isfinite(res)
    res_f = res[finite]
    print(f"Target frame residuals: N={len(res_f)} | mean={np.mean(res_f):.6f} | median={np.median(res_f):.6f} | max={np.max(res_f):.6f}")
    
    fig = plt.figure(figsize=(14,4))
    ax1 = fig.add_subplot(1,2,1)
    ax1.hist(res_f, bins=80)
    ax1.set_title('Target frame residual histogram')
    ax1.set_xlabel('residual')
    ax1.set_ylabel('count')
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    ax2 = fig.add_subplot(1,2,2)
    if len(inl) == len(res):
        ax2.scatter(np.arange(len(res)), res, s=8, label='all')
        ax2.scatter(np.where(~inl)[0], res[~inl], s=12, label='outliers (logged)', alpha=0.8)
        ax2.set_title('Residuals by correspondence index')
        ax2.set_xlabel('corr index')
        ax2.set_ylabel('residual')
        ax2.set_yscale('log')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
    else:
        ax2.text(0.1,0.5,'inliers array size mismatch; skipping scatter split', fontsize=12)
    plt.show()
    
    # Worst track IDs
    k = min(30, len(res_f))
    worst_idx = np.argsort(res_f)[-k:][::-1]
    worst = []
    for wi in worst_idx:
        # map back to original index
        orig_i = np.where(finite)[0][wi]
        tid = int(tids[orig_i]) if orig_i < len(tids) else None
        worst.append((orig_i, tid, float(res[orig_i]), bool(inl[orig_i]) if orig_i < len(inl) else None))
    df_worst_obs = pd.DataFrame(worst, columns=['corr_i','track_id','residual','logged_inlier'])
    print('\nWorst target-frame correspondences by residual:')
    display(df_worst_obs)
    
    # Do these track IDs exist as landmarks?
    if len(lm_idxs_in_values) > 0:
        df_worst_obs['lm_present'] = df_worst_obs['track_id'].apply(lambda t: (t in lm_idxs_in_values) if t is not None else False)
        print('\nWorst correspondences: is there a landmark with the same ID?')
        display(df_worst_obs[['corr_i','track_id','residual','logged_inlier','lm_present']])


## Visualize Edges for a Single Keyframe

This plots the pose position and the landmarks it connects to (optionally only the highest-error edges) so you can *see* the constraint structure.

In [ ]:
# Pick a keyframe to visualize
if len(edges) == 0:
    print("No pose↔landmark edges found; run the graph diagnostics cell first.")
else:
    # Default: keyframe with the largest summed edge error
    viz_kf_idx = max(pose_err_sum.keys(), key=lambda k: pose_err_sum[k]) if len(pose_err_sum) > 0 else edges[0]['pose_idx']
    viz_frame_id = kf_frame_id_by_kf_idx.get(viz_kf_idx, None)

    # Gather edges for this pose
    e_pose = [e for e in edges if e['pose_idx'] == viz_kf_idx and np.isfinite(e['error'])]
    e_pose = sorted(e_pose, key=lambda e: e['error'], reverse=True)

    print(f"Visualizing x_{viz_kf_idx} (frame_id={viz_frame_id})")
    print(f"  degree: {len(e_pose)}")
    print(f"  sum edge error: {pose_err_sum.get(viz_kf_idx, 0.0):.6e}")

    # Only draw a subset (otherwise it can get cluttered)
    max_edges_to_draw = 80
    e_draw = e_pose[:max_edges_to_draw]

    Xi = gtsam.symbol('x', int(viz_kf_idx))
    if not values.exists(Xi):
        print("Pose not found in Values; cannot visualize.")
    else:
        T_x = values.atPose3(Xi).matrix()
        x_t = T_x[:3, 3]

        pts = []
        errs = []
        lids = []
        for e in e_draw:
            Lj = gtsam.symbol('l', int(e['landmark_id']))
            if not values.exists(Lj):
                continue
            p = np.asarray(values.atPoint3(Lj), dtype=float).reshape(3,)
            pts.append(p)
            errs.append(float(e['error']))
            lids.append(int(e['landmark_id']))

        pts = np.asarray(pts, dtype=float)
        errs = np.asarray(errs, dtype=float)

        if pts.shape[0] == 0:
            print("No landmark points for this keyframe found in Values.")
        else:
            import matplotlib.colors as mcolors

            print("Matplotlib backend:", plt.get_backend())

            fig = plt.figure(figsize=(12, 9))
            ax = fig.add_subplot(111, projection='3d')

            # Landmarks colored by edge error
            errs_clip = np.clip(errs, 1e-12, None)
            norm = None
            if float(np.nanmin(errs_clip)) < float(np.nanmax(errs_clip)):
                norm = mcolors.LogNorm(vmin=float(np.nanmin(errs_clip)), vmax=float(np.nanmax(errs_clip)))

            sc = ax.scatter(
                pts[:, 0], pts[:, 1], pts[:, 2],
                c=errs_clip,
                cmap='viridis',
                norm=norm,
                s=30,
                alpha=0.9,
            )
            cb = plt.colorbar(sc, ax=ax, shrink=0.6)
            cb.set_label('edge error' + (' (log)' if norm is not None else ''))

            # Pose location
            ax.scatter([x_t[0]], [x_t[1]], [x_t[2]], c='r', marker='*', s=300, label=f"x_{viz_kf_idx}")

            # Draw line segments for the drawn edges
            for p in pts:
                ax.plot([x_t[0], p[0]], [x_t[1], p[1]], [x_t[2], p[2]], color='k', alpha=0.15, linewidth=1)

            ax.set_title(f"Pose↔Landmark edges for x_{viz_kf_idx} (frame_id={viz_frame_id})  |  showing top-{len(pts)} edges")
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_zlabel('Z')
            ax.legend()
            plt.tight_layout()
            plt.show()


## Keyframe Pose Error: Before vs After Global Optimization

Two views:
- **Internal change**: how much each keyframe pose moved (pre → post global opt)
- **Vs GT (if available)**: translation/rotation error per keyframe before vs after


In [ ]:
def _pose_trans(T: np.ndarray) -> np.ndarray:
    return np.asarray(T[:3, 3], dtype=float).reshape(3,)


def _pose_rot(T: np.ndarray) -> np.ndarray:
    return np.asarray(T[:3, :3], dtype=float).reshape(3, 3)


def _rot_delta_deg(R1: np.ndarray, R2: np.ndarray) -> float:
    R = R1.T @ R2
    tr = float(np.trace(R))
    theta = np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0))
    return float(np.degrees(theta))


kf_idxs = sorted(kf_pose_pre_global.keys())
if len(kf_idxs) == 0:
    print("No keyframe pose bookkeeping found. Re-run the pipeline replay cell.")
else:
    # Plot keyframes against their original frame_id (instead of kf_idx)
    kf_frame_ids = np.asarray([kf_frame_id_by_kf_idx.get(k, k) for k in kf_idxs], dtype=int)

    pre_list = []
    post_list = []
    frame_ids = []
    nobs_list = []
    gt_list = []

    for k in kf_idxs:
        pre = np.asarray(kf_pose_pre_global[k], dtype=float)
        post = np.asarray(kf_pose_post_global.get(k, pre), dtype=float)
        pre_list.append(pre)
        post_list.append(post)
        frame_ids.append(kf_frame_id_by_kf_idx.get(k, None))
        nobs_list.append(kf_num_obs_by_kf_idx.get(k, None))
        gt_list.append(kf_gt_pose_by_kf_idx.get(k, None))

    pre_list = np.asarray(pre_list)
    post_list = np.asarray(post_list)

    # --- Internal pose change (pre -> post) ---
    d_trans = np.array([np.linalg.norm(_pose_trans(a) - _pose_trans(b)) for a, b in zip(pre_list, post_list)], dtype=float)
    d_rot = np.array([_rot_delta_deg(_pose_rot(a), _pose_rot(b)) for a, b in zip(pre_list, post_list)], dtype=float)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax = axes[0]
    ax.plot(kf_frame_ids, d_trans, 'b-o', lw=2)
    ax.set_ylabel('||t_post - t_pre||')
    ax.set_title('Keyframe pose change after global optimization')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(kf_frame_ids, d_rot, 'r-o', lw=2)
    ax.set_ylabel('rot_delta (deg)')
    ax.set_xlabel('frame_id')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Global objective trend over updates ---
    if len(global_kf_idx_updates) > 0:
        fig, ax = plt.subplots(figsize=(14, 4))
        x = np.asarray([kf_frame_id_by_kf_idx.get(k, k) for k in global_kf_idx_updates], dtype=int)
        ax.plot(x, np.asarray(global_obj_error_before, dtype=float), 'k--', lw=1.5, label='obj before update')
        ax.plot(x, np.asarray(global_obj_error_after, dtype=float), 'g-', lw=2, label='obj after update')
        ax.set_yscale('log')
        ax.set_xlabel('frame_id')
        ax.set_ylabel('graph.error(values) (log)')
        ax.set_title('Global objective over time (per keyframe update)')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(x, np.asarray(global_graph_num_factors, dtype=int), 'b-', lw=2, label='# factors')
        ax.plot(x, np.asarray(global_graph_num_values, dtype=int), 'm-', lw=2, label='# values')
        ax.set_xlabel('frame_id')
        ax.set_ylabel('count')
        ax.set_title('Graph growth over time')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

    # --- Vs GT (if available) ---
    has_gt = any(g is not None for g in gt_list)
    if not has_gt:
        print("No GT poses stored for keyframes (kf_gt_pose_by_kf_idx all None). Skipping GT error plots.")
    else:
        # Compute errors only where GT exists
        gt_mask = np.array([g is not None for g in gt_list], dtype=bool)
        kf_idxs_gt = np.array([k for k, m in zip(kf_idxs, gt_mask) if m], dtype=int)
        frame_ids_gt = np.asarray([kf_frame_id_by_kf_idx.get(k, k) for k in kf_idxs_gt], dtype=int)
        pre_gt = pre_list[gt_mask]
        post_gt = post_list[gt_mask]
        gt_arr = np.asarray([g for g in gt_list if g is not None], dtype=float)

        pre_t_err = np.linalg.norm(pre_gt[:, :3, 3] - gt_arr[:, :3, 3], axis=1)
        post_t_err = np.linalg.norm(post_gt[:, :3, 3] - gt_arr[:, :3, 3], axis=1)

        pre_r_err = np.array([_rot_delta_deg(_pose_rot(E), _pose_rot(G)) for E, G in zip(pre_gt, gt_arr)], dtype=float)
        post_r_err = np.array([_rot_delta_deg(_pose_rot(E), _pose_rot(G)) for E, G in zip(post_gt, gt_arr)], dtype=float)

        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        ax = axes[0]
        ax.plot(frame_ids_gt, pre_t_err, 'r--o', lw=2, label='pre')
        ax.plot(frame_ids_gt, post_t_err, 'g-o', lw=2, label='post')
        ax.set_ylabel('translation error')
        ax.set_title('Keyframe pose error vs GT (translation)')
        ax.grid(True, alpha=0.3)
        ax.legend()

        ax = axes[1]
        ax.plot(frame_ids_gt, pre_r_err, 'r--o', lw=2, label='pre')
        ax.plot(frame_ids_gt, post_r_err, 'g-o', lw=2, label='post')
        ax.set_ylabel('rotation error (deg)')
        ax.set_xlabel('frame_id')
        ax.set_title('Keyframe pose error vs GT (rotation)')
        ax.grid(True, alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.show()

        # Improvement plots
        fig, ax = plt.subplots(1, 2, figsize=(16, 4))
        ax0, ax1 = ax

        ax0.bar(frame_ids_gt, pre_t_err - post_t_err)
        ax0.axhline(0, color='k', lw=1)
        ax0.set_title('Translation error improvement (pre - post)')
        ax0.set_xlabel('frame_id')
        ax0.set_ylabel('meters')
        ax0.grid(True, axis='y', alpha=0.3)

        ax1.bar(frame_ids_gt, pre_r_err - post_r_err)
        ax1.axhline(0, color='k', lw=1)
        ax1.set_title('Rotation error improvement (pre - post)')
        ax1.set_xlabel('frame_id')
        ax1.set_ylabel('deg')
        ax1.grid(True, axis='y', alpha=0.3)

        plt.tight_layout()
        plt.show()

    # Extra: correlation (keyframe degree vs pose movement)
    if len(pose_degree) > 0:
        deg_for_kf = np.array([pose_degree.get(k, 0) for k in kf_idxs], dtype=float)
        fig, ax = plt.subplots(1, 2, figsize=(16, 4))
        ax0, ax1 = ax
        ax0.scatter(deg_for_kf, d_trans, s=30, alpha=0.7)
        ax0.set_xscale('log')
        ax0.set_yscale('log')
        ax0.set_xlabel('keyframe degree (# edges)')
        ax0.set_ylabel('||t_post - t_pre||')
        ax0.set_title('Connectivity vs translation change')
        ax0.grid(True, alpha=0.3)

        ax1.scatter(deg_for_kf, d_rot, s=30, alpha=0.7)
        ax1.set_xscale('log')
        ax1.set_yscale('log')
        ax1.set_xlabel('keyframe degree (# edges)')
        ax1.set_ylabel('rot_delta (deg)')
        ax1.set_title('Connectivity vs rotation change')
        ax1.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


## Edge Error Trends Over Iterations (Target Optimization)

If you ran the per-iteration tracking cell (`IterationTrackingOptimizer`), this will:
- extract the edges incident to the *target pose*
- plot their error over LM iterations
- highlight the worst edges/landmarks and whether they converge


In [ ]:
def _edges_incident_to_pose(graph: gtsam.NonlinearFactorGraph, values: gtsam.Values, pose_idx: int):
    Xi = gtsam.symbol('x', int(pose_idx))
    out = []
    for fi in range(graph.size()):
        f = graph.at(fi)
        try:
            keys = list(f.keys())
        except Exception:
            continue
        if Xi not in keys:
            continue
        if len(keys) != 2:
            continue
        try:
            # Only keep x-l edges
            c0, i0 = _sym_chr_idx(keys[0])
            c1, i1 = _sym_chr_idx(keys[1])
        except Exception:
            continue
        if {c0, c1} != {'x', 'l'}:
            continue
        lm_id = i0 if c0 == 'l' else i1
        try:
            err = float(f.error(values))
        except Exception:
            err = float('nan')
        out.append({'factor_index': fi, 'landmark_id': int(lm_id), 'error': err})
    return out


# This requires that the target-iteration cell ran successfully
if 'iteration_tracker' not in globals() or not hasattr(iteration_tracker, 'iteration_values'):
    print("No iteration_tracker.iteration_values found. Run the per-iteration tracking cell first.")
else:
    if len(iteration_tracker.iteration_values) == 0:
        print("iteration_tracker.iteration_values is empty (likely first-frame no-optimization).")
    else:
        if 'target_object_frame_data' in globals():
            target_pose_idx = int(target_object_frame_data.frame_id)
        else:
            # Fallback: try to use FRAME_NUMBER as pose idx (may be wrong for keyframe graph)
            target_pose_idx = int(FRAME_NUMBER)

        g = iteration_tracker.graph
        vals_list = iteration_tracker.iteration_values

        # Collect the set of incident edges at iteration 0 (structure won't change over LM iterations)
        inc0 = _edges_incident_to_pose(g, vals_list[0], target_pose_idx)
        if len(inc0) == 0:
            print(f"No x-l edges incident to x_{target_pose_idx}.")

            # Debug: does this pose appear in the graph at all?
            Xi = gtsam.symbol('x', int(target_pose_idx))
            fac_types = {}
            fac_count = 0
            for fi in range(g.size()):
                f = g.at(fi)
                try:
                    keys = list(f.keys())
                except Exception:
                    continue
                if Xi in keys:
                    fac_count += 1
                    t = type(f).__name__
                    fac_types[t] = fac_types.get(t, 0) + 1
            print(f"Factors involving x_{target_pose_idx}: {fac_count}")
            if fac_count > 0:
                print(f"Factor types involving x_{target_pose_idx}: {fac_types}")

            # Debug: show measurement stats feeding the tracker
            if 'target_object_frame_data' in globals():
                d = target_object_frame_data
                n_inl = int(np.sum(np.asarray(d.reg_inliers, dtype=bool))) if getattr(d, 'reg_inliers', None) is not None and len(d.reg_inliers) > 0 else 0
                n_res = int(len(d.reg_residuals)) if getattr(d, 'reg_residuals', None) is not None else 0
                n_vid = int(len(d.reg_valid_idx)) if getattr(d, 'reg_valid_idx', None) is not None else 0
                print(f"target_object_frame_data: reg_valid_idx={n_vid}, inliers={n_inl}, residuals={n_res}")
                if n_res > 0:
                    r = np.asarray(d.reg_residuals, dtype=float)
                    print(f"residuals: min={float(np.min(r)):.3e}, med={float(np.median(r)):.3e}, mean={float(np.mean(r)):.3e}, max={float(np.max(r)):.3e}")

            print("If you expected edges here, likely causes:")
            print("  - Symbol parsing mismatch (rerun the graph diagnostics cell so _sym_chr_idx is defined correctly).")
            print("  - All measurements were filtered out inside IterationTrackingOptimizer (e.g., residual threshold).")
        else:
            # Build per-iteration error matrix for these factors
            factor_ids = [e['factor_index'] for e in inc0]
            lm_ids = [e['landmark_id'] for e in inc0]

            E = np.full((len(vals_list), len(factor_ids)), np.nan, dtype=float)
            for it, v in enumerate(vals_list):
                for j, fi in enumerate(factor_ids):
                    try:
                        E[it, j] = float(g.at(int(fi)).error(v))
                    except Exception:
                        E[it, j] = np.nan

            # Summary stats per iteration
            mean_e = np.nanmean(E, axis=1)
            med_e = np.nanmedian(E, axis=1)
            max_e = np.nanmax(E, axis=1)

            iters = np.arange(E.shape[0])

            fig, ax = plt.subplots(figsize=(12, 4))
            ax.plot(iters, np.clip(mean_e, 1e-12, None), 'b-o', label='mean edge error')
            ax.plot(iters, np.clip(med_e, 1e-12, None), 'g-o', label='median edge error')
            ax.plot(iters, np.clip(max_e, 1e-12, None), 'r-o', label='max edge error')
            ax.set_yscale('log')
            ax.set_xlabel('iteration')
            ax.set_ylabel('edge error (log)')
            ax.set_title(f"Edges incident to x_{target_pose_idx}: error trend over LM iterations")
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.tight_layout()
            plt.show()

            # Track top-K worst edges (by iteration-0 error)
            e0 = E[0]
            order = np.argsort(-np.nan_to_num(e0, nan=-np.inf))
            topk = order[: min(10, len(order))]

            fig, ax = plt.subplots(figsize=(12, 5))
            for jj in topk:
                ax.plot(iters, np.clip(E[:, jj], 1e-12, None), lw=2, label=f"l_{lm_ids[jj]} (fi={factor_ids[jj]})")
            ax.set_yscale('log')
            ax.set_xlabel('iteration')
            ax.set_ylabel('edge error (log)')
            ax.set_title('Top edges (largest error at iter 0)')
            ax.grid(True, alpha=0.3)
            ax.legend(bbox_to_anchor=(1.02, 1.0), loc='upper left')
            plt.tight_layout()
            plt.show()

            # Initial vs final scatter
            e_final = E[-1]
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.scatter(np.clip(e0, 1e-12, None), np.clip(e_final, 1e-12, None), s=20, alpha=0.7)
            ax.plot([1e-12, max(1e-12, np.nanmax(np.clip(e0, 1e-12, None)))], [1e-12, max(1e-12, np.nanmax(np.clip(e0, 1e-12, None)))], 'k--', lw=1)
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.set_xlabel('edge error at iter 0')
            ax.set_ylabel('edge error at final iter')
            ax.set_title('Edge error improvement (below diagonal = improved)')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            # Distribution shift
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.hist(np.clip(E[0], 1e-12, None), bins=60, alpha=0.6, label='iter 0')
            ax.hist(np.clip(E[-1], 1e-12, None), bins=60, alpha=0.6, label='final')
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.set_xlabel('edge error (log)')
            ax.set_ylabel('count (log)')
            ax.set_title('Incident-edge error distribution: iter 0 vs final')
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.tight_layout()
            plt.show()


In [ ]:
# --- Interactive 3D visualization: optimized landmarks ---
# Uses Plotly for rotate/zoom/pan.

import numpy as np

# Grab optimized landmarks from the most recent optimizer run
if (
    "opt_result" in globals()
    and opt_result is not None
    and getattr(opt_result, "key_points_optimized", None) is not None
):
    _xyz_opt = np.asarray(opt_result.key_points_optimized, dtype=float)
    _ids_opt = getattr(opt_result, "key_points_idx_optimized", None)
    _frame_id = getattr(opt_result, "frame_id", None)
else:
    raise RuntimeError(
        "No optimized landmarks found. Run an optimization cell first so `opt_result.key_points_optimized` exists."
    )

# Handle common shapes: (N, 3) or (T, N, 3)
if _xyz_opt.ndim == 3 and _xyz_opt.shape[-1] == 3:
    _xyz_opt = _xyz_opt[-1]

if _xyz_opt.ndim != 2 or _xyz_opt.shape[1] != 3:
    raise ValueError(f"Expected optimized landmarks shaped (N,3); got {_xyz_opt.shape}")

# Optional overlay: the input 3D points feeding registration (if present in this notebook)
_xyz_in = None
_ids_in = None
if "target_object_frame_data" in globals() and target_object_frame_data is not None:
    if getattr(target_object_frame_data, "reg_cur_3d", None) is not None:
        if len(target_object_frame_data.reg_cur_3d) > 0:
            _xyz_in = np.asarray(target_object_frame_data.reg_cur_3d, dtype=float)
            _ids_in = getattr(target_object_frame_data, "reg_cur_3d_idx", None)

# Filter invalid points
_mask_opt = np.isfinite(_xyz_opt).all(axis=1)
xyz_opt = _xyz_opt[_mask_opt]

if _ids_opt is None:
    ids_opt = np.arange(xyz_opt.shape[0])
else:
    ids_opt = np.asarray(_ids_opt).reshape(-1)[: _xyz_opt.shape[0]][_mask_opt]

try:
    import plotly.graph_objects as go
except Exception as e:
    raise ImportError(
        "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
    ) from e

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=xyz_opt[:, 0],
        y=xyz_opt[:, 1],
        z=xyz_opt[:, 2],
        mode="markers",
        marker=dict(
            size=4,
            color=ids_opt.astype(float),
            colorscale="Turbo",
            opacity=0.9,
            colorbar=dict(title="landmark id" if _ids_opt is not None else "index"),
        ),
        text=[f"id={int(i)}" for i in ids_opt],
        hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
        name="optimized landmarks",
    )
)

if _xyz_in is not None:
    _mask_in = np.isfinite(_xyz_in).all(axis=1)
    xyz_in = _xyz_in[_mask_in]
    if _ids_in is None:
        ids_in = np.arange(xyz_in.shape[0])
    else:
        ids_in = np.asarray(_ids_in).reshape(-1)[: _xyz_in.shape[0]][_mask_in]

    fig.add_trace(
        go.Scatter3d(
            x=xyz_in[:, 0],
            y=xyz_in[:, 1],
            z=xyz_in[:, 2],
            mode="markers",
            marker=dict(size=2, color="rgba(120,120,120,0.55)"),
            text=[f"id={int(i)}" for i in ids_in],
            hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
            name="input reg_cur_3d",
        )
    )

_title = "Optimized landmarks (interactive 3D)"
if _frame_id is not None:
    _title += f" — frame {int(_frame_id)}"

fig.update_layout(
    title=_title,
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
    ),
    legend=dict(x=0.01, y=0.99),
)

fig.show()


In [ ]:
# --- Landmarks across keyframe global optimizations ---
# This uses the snapshots collected during the replay cell (global_landmarks_updates).
# Each slider step corresponds to one keyframe update (kf_graph.update).
#
# UX goals:
# - Keep a fixed scene scale (axis ranges) across slider steps so you can see changes.
# - Keep the camera/view stable across slider steps, while still allowing interactive rotate/zoom.
#   Plotly trick: set layout.uirevision to a constant.

OBJ_ID = 0
PLOT_WIDTH = 1200
PLOT_HEIGHT = 850
FIX_AXIS_RANGES = True
ASPECT_MODE = "cube"  # "data" or "cube"

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Re-run the replay cell (Cell 10).")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. Re-run the replay cell (Cell 10) to populate it.")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Establish global color range over ids
        _all_ids = []
        _all_xyz = []
        for u in updates:
            ids = u.get("ids", None)
            xyz = u.get("xyz", None)
            if ids is not None:
                ids = np.asarray(ids, dtype=int).reshape(-1)
                if ids.size > 0:
                    _all_ids.append(ids)
            if xyz is not None:
                xyz = np.asarray(xyz, dtype=float)
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])

        if len(_all_ids) == 0:
            raise RuntimeError("No landmark ids found in global_landmarks_updates.")
        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in global_landmarks_updates.")

        all_ids = np.concatenate(_all_ids, axis=0)
        cmin = float(np.min(all_ids))
        cmax = float(np.max(all_ids))

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges (so switching frames won't auto-rescale)
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            # Pad and optionally make cubic
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )

        # Baseline snapshot (first keyframe update)
        base = updates[0]
        xyz0 = np.asarray(base["xyz"], dtype=float)
        ids0 = np.asarray(base["ids"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color="rgba(140,140,140,0.45)"),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # One trace per update; slider toggles visibility
        for k, u in enumerate(updates):
            xyz = np.asarray(u["xyz"], dtype=float)
            ids = np.asarray(u["ids"], dtype=int).reshape(-1)
            m = np.isfinite(xyz).all(axis=1)
            kf_idx = int(u.get("kf_idx", -1))
            frame_id = int(u.get("frame_id", -1))

            fig.add_trace(
                go.Scatter3d(
                    x=xyz[m, 0],
                    y=xyz[m, 1],
                    z=xyz[m, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=ids[m].astype(float),
                        colorscale="Turbo",
                        cmin=cmin,
                        cmax=cmax,
                        opacity=0.9,
                        colorbar=dict(title="landmark id"),
                    ),
                    text=[f"id={int(i)}" for i in ids[m]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"kf_idx={kf_idx}, frame_id={frame_id}",
                    visible=(k == 0),
                )
            )

        steps = []
        for k, u in enumerate(updates):
            vis = [True] + [False] * len(updates)  # baseline always on
            vis[1 + k] = True
            kf_idx = int(u.get("kf_idx", -1))
            frame_id = int(u.get("frame_id", -1))
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Global optimized landmarks over keyframes (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Global optimized landmarks over keyframes (obj {OBJ_ID}) — kf_idx={int(updates[0].get('kf_idx', -1))}, frame_id={int(updates[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            # Keeps user-driven camera/zoom across slider steps
            uirevision=f"kf_landmarks_obj_{OBJ_ID}",
        )

        fig.show()

In [ ]:
# --- Landmark error trends over KEYFRAME-GRAPH updates ---
# We compute how each landmark's position would change if we replace the estimated pose with GT pose:
#   p_obj -> p_cam(est) -> p_obj(gt)
# Error per landmark = ||p_obj(gt) - p_obj(est)||
#
# This matches the idea: transform landmark to camera using estimated pose, then use GT pose to map back
# into the (GT-aligned) object frame ("first pose" gauge).

OBJ_ID = 0
TOP_K = 400

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Re-run the replay cell (Cell 10).")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Check required fields
        _missing = []
        for k in ["pose_est", "pose_gt", "xyz", "ids", "frame_id", "kf_idx"]:
            if any(k not in u for u in updates):
                _missing.append(k)
        if len(_missing) > 0:
            print(
                "Missing fields in global_landmarks_updates: "
                + ", ".join(_missing)
                + ". Re-run replay (Cell 10) and target tracking (Cell 12) after the latest notebook edits."
            )
        else:
            frame_ids = np.array([int(u.get("frame_id", -1)) for u in updates], dtype=int)
            kf_idxs = np.array([int(u.get("kf_idx", -1)) for u in updates], dtype=int)

            # Per-update dict: landmark_id -> error
            err_by_update = []
            for u in updates:
                T_est = u.get("pose_est", None)
                T_gt = u.get("pose_gt", None)
                xyz = np.asarray(u.get("xyz", np.zeros((0, 3))), dtype=float)
                ids = np.asarray(u.get("ids", np.zeros((0,), dtype=int)), dtype=int).reshape(-1)

                if T_est is None or T_gt is None:
                    err_by_update.append({})
                    continue

                T_est = np.asarray(T_est, dtype=float)
                T_gt = np.asarray(T_gt, dtype=float)

                if xyz.ndim != 2 or xyz.shape[1] != 3:
                    err_by_update.append({})
                    continue

                m = np.isfinite(xyz).all(axis=1)
                xyz = xyz[m]
                ids = ids[m]

                if xyz.shape[0] == 0:
                    err_by_update.append({})
                    continue

                # p_obj -> p_cam(est)
                xyz_cam = transform_pts(T_est, xyz)
                # p_cam -> p_obj(gt)
                xyz_gt_obj = transform_pts(inverse_SE3(T_gt), xyz_cam)

                e = np.linalg.norm(xyz_gt_obj - xyz, axis=1)
                err_by_update.append({int(i): float(v) for i, v in zip(ids.tolist(), e.tolist())})

            # Summary stats per update
            mean_e = []
            med_e = []
            p90_e = []
            max_e = []
            for d in err_by_update:
                if len(d) == 0:
                    mean_e.append(np.nan)
                    med_e.append(np.nan)
                    p90_e.append(np.nan)
                    max_e.append(np.nan)
                    continue
                vals = np.asarray(list(d.values()), dtype=float)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    mean_e.append(np.nan)
                    med_e.append(np.nan)
                    p90_e.append(np.nan)
                    max_e.append(np.nan)
                else:
                    mean_e.append(float(np.mean(vals)))
                    med_e.append(float(np.median(vals)))
                    p90_e.append(float(np.percentile(vals, 90)))
                    max_e.append(float(np.max(vals)))

            fig, ax = plt.subplots(figsize=(14, 4))
            ax.plot(frame_ids, mean_e, "b-o", lw=2, label="mean")
            ax.plot(frame_ids, med_e, "g-o", lw=2, label="median")
            ax.plot(frame_ids, p90_e, "m-o", lw=2, label="p90")
            ax.plot(frame_ids, max_e, "r-o", lw=2, label="max")
            ax.set_yscale("log")
            ax.set_xlabel("frame_id (keyframes)")
            ax.set_ylabel("||p_obj(gt) - p_obj(est)|| (m)")
            ax.set_title("Landmark error trend across keyframe-graph updates")
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.tight_layout()
            plt.show()

            # Choose TOP_K landmarks by mean error across updates
            all_ids = sorted({lid for d in err_by_update for lid in d.keys()})
            if len(all_ids) == 0:
                print("No per-landmark errors computed (likely missing GT for updates).")
            else:
                id_to_col = {lid: j for j, lid in enumerate(all_ids)}
                E = np.full((len(updates), len(all_ids)), np.nan, dtype=float)
                for i, d in enumerate(err_by_update):
                    for lid, v in d.items():
                        E[i, id_to_col[lid]] = float(v)

                mean_per_id = np.nanmean(E, axis=0)
                order = np.argsort(-np.nan_to_num(mean_per_id, nan=-np.inf))
                top_cols = order[: min(TOP_K, len(order))]
                top_ids = [all_ids[j] for j in top_cols]

                fig, ax = plt.subplots(figsize=(14, 6))
                for lid, j in zip(top_ids, top_cols):
                    ax.plot(frame_ids, E[:, j], lw=2, label=f"l_{lid}")
                ax.set_yscale("log")
                ax.set_xlabel("frame_id (keyframes)")
                ax.set_ylabel("error (m)")
                ax.set_title(f"Top-{len(top_ids)} landmarks by mean error (pose-corrected vs est)")
                ax.grid(True, alpha=0.3)
                ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left")
                plt.tight_layout()
                plt.show()

                # Heatmap for the same top landmarks
                fig, ax = plt.subplots(figsize=(14, 0.35 * len(updates) + 2))
                im = ax.imshow(
                    np.log10(np.clip(E[:, top_cols], 1e-12, None)),
                    aspect="auto",
                    interpolation="nearest",
                )
                ax.set_yticks(np.arange(len(updates)))
                ax.set_yticklabels([str(int(fid)) for fid in frame_ids])
                ax.set_xticks(np.arange(len(top_ids)))
                ax.set_xticklabels([f"l_{lid}" for lid in top_ids], rotation=45, ha="right")
                ax.set_xlabel("landmark id")
                ax.set_ylabel("keyframe frame_id")
                ax.set_title("log10 landmark error heatmap (top landmarks)")
                plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
                plt.tight_layout()
                plt.show()

In [ ]:
# --- Why does the map (landmarks) barely move? Diagnostics ---
# This section quantifies:
# 1) Map delta between successive keyframe-graph updates (||Δ landmark||)
# 2) Measurement residual in camera frame (|| z_cam - (T_cam_obj * p_obj) ||)
# 3) Correlation with landmark observation count / degree

OBJ_ID = 0
ONLY_INLIERS = True

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Re-run the replay cell (Cell 10).")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

    if len(updates) < 2:
        print("Need at least 2 keyframe updates to analyze map changes. Re-run replay/tracking.")
    else:
        # --- Map delta across updates ---
        map_mean = []
        map_med = []
        map_p90 = []
        map_max = []
        x_frame = []

        # --- Measurement residual (camera frame) ---
        meas_mean = []
        meas_med = []
        meas_p90 = []
        meas_max = []

        # Landmark observation counts (from measurements in each keyframe)
        obs_count = {}

        for u in updates:
            ids = np.asarray(u.get("meas_ids", []), dtype=int).reshape(-1)
            inl = np.asarray(u.get("meas_inliers", np.ones_like(ids, dtype=bool)), dtype=bool).reshape(-1)
            if ONLY_INLIERS and ids.size > 0 and inl.size == ids.size:
                ids = ids[inl]
            for lid in ids.tolist():
                obs_count[lid] = obs_count.get(lid, 0) + 1

        for k in range(1, len(updates)):
            u0 = updates[k - 1]
            u1 = updates[k]

            fid = int(u1.get("frame_id", -1))
            x_frame.append(fid)

            ids0 = np.asarray(u0.get("ids", []), dtype=int).reshape(-1)
            xyz0 = np.asarray(u0.get("xyz", np.zeros((0, 3))), dtype=float)
            ids1 = np.asarray(u1.get("ids", []), dtype=int).reshape(-1)
            xyz1 = np.asarray(u1.get("xyz", np.zeros((0, 3))), dtype=float)

            if xyz0.ndim != 2 or xyz1.ndim != 2 or xyz0.shape[1] != 3 or xyz1.shape[1] != 3:
                map_mean.append(np.nan)
                map_med.append(np.nan)
                map_p90.append(np.nan)
                map_max.append(np.nan)
            else:
                m0 = np.isfinite(xyz0).all(axis=1)
                m1 = np.isfinite(xyz1).all(axis=1)
                ids0 = ids0[m0]
                xyz0 = xyz0[m0]
                ids1 = ids1[m1]
                xyz1 = xyz1[m1]

                idx0 = {int(i): j for j, i in enumerate(ids0.tolist())}
                common = [int(i) for i in ids1.tolist() if int(i) in idx0]

                if len(common) == 0:
                    map_mean.append(np.nan)
                    map_med.append(np.nan)
                    map_p90.append(np.nan)
                    map_max.append(np.nan)
                else:
                    p0 = np.stack([xyz0[idx0[i]] for i in common], axis=0)
                    idx1 = {int(i): j for j, i in enumerate(ids1.tolist())}
                    p1 = np.stack([xyz1[idx1[i]] for i in common], axis=0)
                    d = np.linalg.norm(p1 - p0, axis=1)
                    map_mean.append(float(np.mean(d)))
                    map_med.append(float(np.median(d)))
                    map_p90.append(float(np.percentile(d, 90)))
                    map_max.append(float(np.max(d)))

            # Measurement residual at this update
            T_est = u1.get("pose_est", None)
            meas_xyz = np.asarray(u1.get("meas_xyz_cam", np.zeros((0, 3))), dtype=float)
            meas_ids = np.asarray(u1.get("meas_ids", []), dtype=int).reshape(-1)
            meas_inl = np.asarray(u1.get("meas_inliers", np.ones_like(meas_ids, dtype=bool)), dtype=bool).reshape(-1)

            if T_est is None or meas_xyz.ndim != 2 or meas_xyz.shape[1] != 3 or meas_ids.size == 0:
                meas_mean.append(np.nan)
                meas_med.append(np.nan)
                meas_p90.append(np.nan)
                meas_max.append(np.nan)
            else:
                T_est = np.asarray(T_est, dtype=float)

                if ONLY_INLIERS and meas_inl.size == meas_ids.size:
                    m = meas_inl & np.isfinite(meas_xyz).all(axis=1)
                else:
                    m = np.isfinite(meas_xyz).all(axis=1)

                meas_xyz = meas_xyz[m]
                meas_ids = meas_ids[m]

                # Build id->map point lookup from this update's map snapshot
                ids_map = np.asarray(u1.get("ids", []), dtype=int).reshape(-1)
                xyz_map = np.asarray(u1.get("xyz", np.zeros((0, 3))), dtype=float)
                mm = np.isfinite(xyz_map).all(axis=1)
                ids_map = ids_map[mm]
                xyz_map = xyz_map[mm]
                idx_map = {int(i): j for j, i in enumerate(ids_map.tolist())}

                pred = []
                z = []
                for pid, pz in zip(meas_ids.tolist(), meas_xyz):
                    j = idx_map.get(int(pid), None)
                    if j is None:
                        continue
                    pred.append(transform_pts(T_est, xyz_map[j:j+1])[0])
                    z.append(pz)

                if len(pred) == 0:
                    meas_mean.append(np.nan)
                    meas_med.append(np.nan)
                    meas_p90.append(np.nan)
                    meas_max.append(np.nan)
                else:
                    pred = np.asarray(pred, dtype=float)
                    z = np.asarray(z, dtype=float)
                    r = np.linalg.norm(z - pred, axis=1)
                    meas_mean.append(float(np.mean(r)))
                    meas_med.append(float(np.median(r)))
                    meas_p90.append(float(np.percentile(r, 90)))
                    meas_max.append(float(np.max(r)))

        # Plot map-delta and measurement residual trends
        fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

        ax = axes[0]
        ax.plot(x_frame, map_mean, "b-o", lw=2, label="mean ||Δmap||")
        ax.plot(x_frame, map_med, "g-o", lw=2, label="median")
        ax.plot(x_frame, map_p90, "m-o", lw=2, label="p90")
        ax.plot(x_frame, map_max, "r-o", lw=2, label="max")
        ax.set_yscale("log")
        ax.set_ylabel("meters")
        ax.set_title("Map update magnitude per keyframe update")
        ax.grid(True, alpha=0.3)
        ax.legend()

        ax = axes[1]
        ax.plot(x_frame, meas_mean, "b-o", lw=2, label="mean ||z - pred||")
        ax.plot(x_frame, meas_med, "g-o", lw=2, label="median")
        ax.plot(x_frame, meas_p90, "m-o", lw=2, label="p90")
        ax.plot(x_frame, meas_max, "r-o", lw=2, label="max")
        ax.set_yscale("log")
        ax.set_xlabel("frame_id (keyframes)")
        ax.set_ylabel("meters")
        ax.set_title("3D measurement residual in camera frame")
        ax.grid(True, alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.show()

        # Correlate per-landmark observation count vs (a) mean error (from Cell 43) and (b) movement
        # Movement: total displacement from first to last snapshot for landmarks present in both.
        first = updates[0]
        last = updates[-1]
        ids0 = np.asarray(first.get("ids", []), dtype=int).reshape(-1)
        xyz0 = np.asarray(first.get("xyz", np.zeros((0, 3))), dtype=float)
        ids1 = np.asarray(last.get("ids", []), dtype=int).reshape(-1)
        xyz1 = np.asarray(last.get("xyz", np.zeros((0, 3))), dtype=float)

        m0 = np.isfinite(xyz0).all(axis=1)
        m1 = np.isfinite(xyz1).all(axis=1)
        ids0, xyz0 = ids0[m0], xyz0[m0]
        ids1, xyz1 = ids1[m1], xyz1[m1]
        idx0 = {int(i): j for j, i in enumerate(ids0.tolist())}
        idx1 = {int(i): j for j, i in enumerate(ids1.tolist())}
        common = [i for i in idx0.keys() if i in idx1]

        if len(common) > 0:
            move = []
            deg = []
            for lid in common:
                move.append(float(np.linalg.norm(xyz1[idx1[lid]] - xyz0[idx0[lid]])))
                deg.append(float(obs_count.get(lid, 0)))

            fig, ax = plt.subplots(figsize=(6, 5))
            ax.scatter(deg, np.clip(move, 1e-12, None), s=20, alpha=0.6)
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("# keyframe observations (log)")
            ax.set_ylabel("total map displacement (m, log)")
            ax.set_title("Landmark degree vs map movement")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        # Print current formulation knobs (so it's obvious why map may be stiff)
        try:
            cfg = pipeline_cfg.global_optimizer.params
            print("\n=== Formulation knobs (global optimizer) ===")
            print(f"use_between_factor:      {bool(getattr(cfg, 'use_between_factor', False))}")
            print(f"between_noise_param:     {getattr(cfg, 'between_noise_param', None)}")
            print(f"landmark_noise_param:    {getattr(cfg, 'landmark_noise_param', None)}")
            print(f"landmark_use_robust:     {bool(getattr(cfg, 'landmark_use_robust', True))}")
            print(f"landmark_huber_k:        {float(getattr(cfg, 'landmark_huber_k', 1.345))}")
            print(f"landmark_sigma_scale_by: {getattr(cfg, 'landmark_sigma_scale_by', 'none')}")
        except Exception as _e:
            print(f"(Could not print formulation knobs: {_e})")


In [ ]:
# --- Map correctability: landmark re-observation + parallax proxy ---
# If many landmarks are only observed in ONE keyframe, the backend cannot refine them (unconstrained).
# If landmarks are observed multiple times but with near-zero parallax, depth/structure refinement is weak.

OBJ_ID = 0
ONLY_INLIERS = True
MIN_KF_OBS = 2
PARALLAX_THRESH_DEG = 2.0

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Re-run the replay cell (Cell 10).")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

    if len(updates) == 0:
        print(f"No updates for OBJ_ID={OBJ_ID}.")
    else:
        obs_kf_set = {}      # lid -> set(kf_idx)
        bearings_by_lid = {} # lid -> list of unit bearing vectors (camera frame)

        for u in updates:
            kf_idx = int(u.get("kf_idx", -1))
            ids = np.asarray(u.get("meas_ids", []), dtype=int).reshape(-1)
            xyz = np.asarray(u.get("meas_xyz_cam", np.zeros((0, 3))), dtype=float)
            inl = np.asarray(u.get("meas_inliers", np.ones((ids.shape[0],), dtype=bool)), dtype=bool).reshape(-1)

            if xyz.ndim != 2 or xyz.shape[1] != 3:
                continue

            n = int(min(ids.shape[0], xyz.shape[0], inl.shape[0]))
            ids = ids[:n]
            xyz = xyz[:n]
            inl = inl[:n]

            m = np.isfinite(xyz).all(axis=1)
            if ONLY_INLIERS:
                m = m & inl

            ids = ids[m]
            xyz = xyz[m]

            for lid, z in zip(ids.tolist(), xyz):
                lid = int(lid)
                zn = float(np.linalg.norm(z))
                if zn <= 1e-9:
                    continue
                b = (z / zn).astype(float)
                obs_kf_set.setdefault(lid, set()).add(kf_idx)
                bearings_by_lid.setdefault(lid, []).append(b)

        if len(obs_kf_set) == 0:
            print("No landmark observations found in global_landmarks_updates.")
        else:
            lids = np.array(sorted(obs_kf_set.keys()), dtype=int)
            deg = np.array([len(obs_kf_set[int(l)]) for l in lids], dtype=int)

            # Degree distribution
            total = int(deg.size)
            frac_deg1 = float(np.mean(deg == 1))
            frac_ge2 = float(np.mean(deg >= 2))
            frac_ge3 = float(np.mean(deg >= 3))

            print("=== Landmark keyframe re-observation counts ===")
            print(f"# landmarks observed at least once: {total}")
            print(f"fraction degree==1: {frac_deg1*100:.1f}%")
            print(f"fraction degree>=2: {frac_ge2*100:.1f}%")
            print(f"fraction degree>=3: {frac_ge3*100:.1f}%")

            fig, ax = plt.subplots(figsize=(10, 4))
            bins = np.arange(1, int(np.max(deg)) + 2)
            ax.hist(deg, bins=bins, alpha=0.8, color="tab:blue")
            ax.set_yscale("log")
            ax.set_xlabel("# keyframes observed")
            ax.set_ylabel("count (log)")
            ax.set_title("Landmark re-observation count across keyframes")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            # Parallax proxy: max bearing angle between any two observations
            parallax_deg = np.full((lids.shape[0],), np.nan, dtype=float)
            for i, lid in enumerate(lids.tolist()):
                if len(obs_kf_set[lid]) < 2:
                    continue
                B = np.asarray(bearings_by_lid.get(lid, []), dtype=float)
                if B.ndim != 2 or B.shape[0] < 2:
                    continue
                # Pairwise dots -> max angle = arccos(min dot)
                G = B @ B.T
                iu = np.triu_indices(G.shape[0], k=1)
                if iu[0].size == 0:
                    continue
                min_dot = float(np.min(np.clip(G[iu], -1.0, 1.0)))
                parallax_deg[i] = float(np.degrees(np.arccos(min_dot)))

            good_parallax = np.isfinite(parallax_deg)
            print("\n=== Parallax proxy (bearing angle change) ===")
            if not np.any(good_parallax):
                print("No multi-view parallax could be computed (need >=2 observations per landmark).")
            else:
                vals = parallax_deg[good_parallax]
                print(f"computed for {int(vals.size)} landmarks")
                print(
                    "parallax_deg percentiles: "
                    f"p10={np.percentile(vals,10):.2f}, p50={np.percentile(vals,50):.2f}, p90={np.percentile(vals,90):.2f}"
                )

                fig, ax = plt.subplots(figsize=(10, 4))
                ax.hist(vals, bins=60, alpha=0.85, color="tab:green")
                ax.axvline(PARALLAX_THRESH_DEG, color="k", ls="--", lw=1, alpha=0.6, label=f"thresh={PARALLAX_THRESH_DEG}°")
                ax.set_xlabel("max bearing angle change (deg)")
                ax.set_ylabel("count")
                ax.set_title("Landmark parallax proxy across keyframes")
                ax.grid(True, alpha=0.3)
                ax.legend()
                plt.tight_layout()
                plt.show()

                # Filter suggestion
                keep = (deg >= MIN_KF_OBS) & (parallax_deg >= PARALLAX_THRESH_DEG)
                keep &= np.isfinite(parallax_deg)
                print("\n=== Suggested 'refinable landmark' filter ===")
                print(f"MIN_KF_OBS={MIN_KF_OBS}, PARALLAX_THRESH_DEG={PARALLAX_THRESH_DEG}")
                print(f"kept: {int(np.sum(keep))}/{total} ({100*float(np.mean(keep)):.1f}%)")

                fig, ax = plt.subplots(figsize=(6, 5))
                ax.scatter(deg[good_parallax], vals, s=15, alpha=0.6)
                ax.set_xscale("log")
                ax.set_xlabel("# keyframes observed (log)")
                ax.set_ylabel("max parallax (deg)")
                ax.set_title("Degree vs parallax")
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()


In [ ]:
# --- Anisotropic covariance helper (depth + tracking uncertainty) ---
# For BearingRangeFactor3D, a physically-plausible anisotropic noise is:
# - bearing sigma (rad) ~ sigma_pix / f
# - range sigma (m)     ~ depth / registration uncertainty
#
# This cell inspects the distributions of:
# - meas_uncertainties (often correlates with tracking / pixel uncertainty)
# - meas_residuals     (often correlates with depth/registration quality)
# and computes suggested (sigma_bearing, sigma_range) values.

OBJ_ID = 0
ONLY_INLIERS = True

# If your uncertainty is already in pixels, set PIX_SCALE=1.
# If it's some normalized score, PIX_SCALE rescales it to "pixel-like" units.
PIX_SCALE = 1.0

# Range sigma heuristic: sigma_range = max(RANGE_MIN_M, RANGE_SCALE * residual)
RANGE_SCALE = 1.0
RANGE_MIN_M = 0.005

BEARING_MIN_RAD = 1e-4

# Intrinsics (use target_fd if present; otherwise fallback to typical HO3D intrinsics)
if "target_fd" in globals() and target_fd is not None and "intrinsics" in target_fd:
    K = np.asarray(target_fd["intrinsics"], dtype=float)
else:
    K = np.array([[615.377, 0, 320], [0, 615.377, 240], [0, 0, 1]], dtype=float)

f = float(0.5 * (K[0, 0] + K[1, 1]))

if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Re-run the replay cell (Cell 10).")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

    if len(updates) == 0:
        print(f"No updates for OBJ_ID={OBJ_ID}.")
    else:
        pix = []
        res = []
        rng = []

        for u in updates:
            ids = np.asarray(u.get("meas_ids", []), dtype=int).reshape(-1)
            xyz = np.asarray(u.get("meas_xyz_cam", np.zeros((0, 3))), dtype=float)
            inl = np.asarray(u.get("meas_inliers", np.ones((ids.shape[0],), dtype=bool)), dtype=bool).reshape(-1)
            unc = np.asarray(u.get("meas_uncertainties", np.zeros((ids.shape[0],), dtype=float)), dtype=float).reshape(-1)
            rr = np.asarray(u.get("meas_residuals", np.zeros((ids.shape[0],), dtype=float)), dtype=float).reshape(-1)

            if xyz.ndim != 2 or xyz.shape[1] != 3:
                continue

            n = int(min(ids.shape[0], xyz.shape[0], inl.shape[0], unc.shape[0], rr.shape[0]))
            xyz = xyz[:n]
            inl = inl[:n]
            unc = unc[:n]
            rr = rr[:n]

            m = np.isfinite(xyz).all(axis=1) & np.isfinite(unc) & np.isfinite(rr)
            if ONLY_INLIERS:
                m = m & inl

            if not np.any(m):
                continue

            xyz = xyz[m]
            unc = unc[m]
            rr = rr[m]

            pix.append(unc)
            res.append(rr)
            rng.append(np.linalg.norm(xyz, axis=1))

        if len(pix) == 0:
            print("No measurements found with uncertainties/residuals. Re-run replay (Cell 10).")
        else:
            pix = np.concatenate(pix).astype(float)
            res = np.concatenate(res).astype(float)
            rng = np.concatenate(rng).astype(float)

            # Convert to sigmas
            sigma_pix = np.clip(pix * float(PIX_SCALE), 0.0, np.inf)
            sigma_bearing = np.maximum(float(BEARING_MIN_RAD), sigma_pix / max(1e-6, f))
            sigma_range = np.maximum(float(RANGE_MIN_M), float(RANGE_SCALE) * np.abs(res))

            print("=== Anisotropic sigma stats (for BearingRangeFactor3D) ===")
            print(f"focal length used: f={f:.2f}")
            print(
                "sigma_pix percentiles: "
                f"p10={np.percentile(sigma_pix,10):.3g}, p50={np.percentile(sigma_pix,50):.3g}, p90={np.percentile(sigma_pix,90):.3g}"
            )
            print(
                "sigma_bearing(rad) percentiles: "
                f"p10={np.percentile(sigma_bearing,10):.3g}, p50={np.percentile(sigma_bearing,50):.3g}, p90={np.percentile(sigma_bearing,90):.3g}"
            )
            print(
                "sigma_range(m) percentiles: "
                f"p10={np.percentile(sigma_range,10):.3g}, p50={np.percentile(sigma_range,50):.3g}, p90={np.percentile(sigma_range,90):.3g}"
            )

            # Helpful derived quantity: lateral metric uncertainty from pixel noise: z/f * sigma_pix
            sigma_lateral = rng / max(1e-6, f) * sigma_pix
            print(
                "sigma_lateral(m) ~ (range/f)*sigma_pix percentiles: "
                f"p10={np.percentile(sigma_lateral,10):.3g}, p50={np.percentile(sigma_lateral,50):.3g}, p90={np.percentile(sigma_lateral,90):.3g}"
            )

            # Plots
            fig, axes = plt.subplots(1, 3, figsize=(16, 4))

            ax = axes[0]
            ax.hist(np.clip(sigma_pix, 0, np.percentile(sigma_pix, 99)), bins=60, alpha=0.85)
            ax.set_title("meas_uncertainties (scaled)")
            ax.set_xlabel("sigma_pix (arb/pix)")
            ax.set_ylabel("count")
            ax.grid(True, alpha=0.3)

            ax = axes[1]
            ax.hist(np.clip(sigma_bearing, 0, np.percentile(sigma_bearing, 99)), bins=60, alpha=0.85)
            ax.set_title("bearing sigma")
            ax.set_xlabel("sigma_bearing (rad)")
            ax.grid(True, alpha=0.3)

            ax = axes[2]
            ax.hist(np.clip(sigma_range, 0, np.percentile(sigma_range, 99)), bins=60, alpha=0.85)
            ax.set_title("range sigma")
            ax.set_xlabel("sigma_range (m)")
            ax.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            # Suggested constant anisotropic noise (if you want a single fixed sigma for all measurements)
            sig_b = float(np.median(sigma_bearing))
            sig_r = float(np.median(sigma_range))
            print("\nSuggested fixed landmark_noise_param (bearing, bearing, range):")
            print([sig_b, sig_b, sig_r])
            print(
                "Note: a *better* model is per-measurement anisotropic sigma (bearing from sigma_pix/f, range from residual/depth)."
            )
